# Deney 06 — Render Pipeline (Notebook 01+02+03 çıktılarını birleştirir)

**Amaç:** Onaylanmış an + script + seslendirmeyi kullanarak dikey (9:16) bir YouTube Shorts videosu üretmek: kaynak aralığı kes, dikey kadraj uygula, seslendirmeyi orijinal sesle miksle (ducking), altyazı yak, kaliteli kodlama ile dışa aktar.

**Çalıştırma sırası:** Notebook 01 → an onayı → Notebook 02 → script onayı → Notebook 03 → seslendirme onayı → **bu notebook**. Alternatif olarak `00_youtube_automation_master.ipynb` tek başına tüm fazları çalıştırabilir.

**Bu notebook hiçbir yere yükleme/yayınlama yapmaz.** Çıktı sadece `results/render/final_short.mp4` olarak yerelde/Colab oturumunda kalır.

**Tek kaynak politikası (düzeltildi):** Bu notebook artık ffmpeg filtergraph'larını, altyazı üretimini, doğrulamayı ve onay kapısı mantığını **kendi içinde tekrar tanımlamaz**. Hepsi `local_pipeline/render_utils.py` içindeki kanonik fonksiyonlardan gelir — master notebook (`00_...`) ve `verify_local.py` de **aynı** fonksiyonları çağırır, yani üç ayrı kopya yoktur. Modül aşağıdaki bootstrap hücresiyle yüklenir (repo → Drive → `PROJECT_REPO_URL` → gömülü hash-doğrulamalı yedek).

**Onay politikası (üretimden sonra, içeriğe bağlı):** `run_render_pipeline()` **onay argümanı almaz** ve asla `approved` üretemez — yeni render her zaman `review_required` (teknik doğrulama başarısızsa `failed`) olur. Onay, videoyu **izledikten sonra** bölüm 5'teki ayrı hücrede, dosyanın SHA-256'sı doğrulanarak verilir. Yeniden render alırsan hash değişir ve önceki onay otomatik geçersizleşir. `render_validation` bir **teknik** kontrol sonucudur, insan onayı değildir.

In [ ]:
!pip -q install imageio-ffmpeg faster-whisper pandas > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, json
from pathlib import Path

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
RENDER_DIR = RESULTS_DIR / 'render'
RENDER_DIR.mkdir(parents=True, exist_ok=True)

STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}
print('Sonuc dizini:', RESULTS_DIR)


## `local_pipeline` kanonik modüllerini yükle (tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['render_utils', 'video_analysis', 'approval']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'render_utils': {
        'sha256': '2d7d4e0ffff2d5d14c61aea86a1ca4f8781b882486ff03335b99b26ff8f21f85',
        'encoding': 'zlib+b85',
        'data': (
            'c-rl~>w4QpwkY_YPf@MR*&q{;mgGy^bTj=aw%qZ=u{@TN>}*Ac0!ffW1OjYaB-1+lzVin2=RD3l&3S=&g;|%XLRA5vWXIh-y?1Bt'
            'PAH&ip)RXdt^3;B+j|$5c`%7X@$%(r9WF!`CbKZ>ucLJsM@cA_VZ08rT;$6j3uj`o5&KWZ+-R&C4WjiXnTYPqGMX+sQG(BM5hNn~'
            'YZ+Y!ahMdoc%H_=q#r@YHTF&lF-_MSk<LZ2jB*jDi)h+|PBU>GX3=~zj?-xn57wIw!xhnLon}SO%~$EHgh9`uESwhcM$FR;UW9@='
            '$w`QOm0pFS2=k&h=ycws&@)-UjIvOa>o^T&xqxo#auP@Rl199tCIj)R5NQHnvk)6G(0LS7o0+)DA^_SkAD8sOK){n2mM<^Y>o|h-'
            'O3Pwh7GfO~%Ry&vZ?7}Y(v=vG=Vei5;dm@(1z?l32#P38@=ixSTL$?ujwb5Q-|{q3zq3&N%9mvk#rm&uvd+?JnCI$AvBWvRr1;CT'
            'pa_d-6{_#5zkPxK{*Wdix5EL$w^g&Z@MouUdUka7<LUUPqt`zkpNf&V?Fjg1FMJOe0=<(Wi1&Q4Hw&-BI9=m|@j4Es;WCY3b?{Xd'
            'UPs{#;13`Mpwphu5P~&~`WpWhdr^{?^LaFlusaMdz=!hyR+(Ny!&y1Sg&O0;HUMG^Bm42&vsY)YkH<fM_v-ZR@w@R4Cohg)(-^%X'
            'N;aM^ynF?FgFjZ`EGk#{D-N<n*uQ?*hs~Iz9sqiN^!Du4$(!-ncgH^*kKevKdHL$KF;{OE&Vw>8yrFzR4`ED*1!F+m90mq25B%!w'
            'JcWrQ!77C3-iz?J;HOfY21)L@ZSv>?w)y(dcoD!<ufy>)jnix#<T=9lx9|4hpO^5D31SlD4eegObla_h_v4#1n_-hj&hsf8a(w=9'
            '-+GZPqGWsx&kqmH7l7koHigf^o0mbHhsL`&EDE3y`8t>a?m+|Dam)Yl7O)=UbUp_31f0w7TF=9__52~orePeHxj22^x}f|2@C7}5'
            '`mqaomX*8Lb6|ojGC-(l7Om-E!1}G$h@)NIf?B-@!d04xGx&axZNkJC@0LNBY_h<7_EVIH*(Nt1{jkY{EDGSn*Z{vv%RDq2{tUD>'
            '%+lN!FS1|(#H@hb^~Luv=3l*ByawtCKWlJaWjHKC1^9jtZ{TA%k`NB)1`t5E?zvlxcv8cK)93wn;i81jgQA$%(P9}KK7N8d_8;y)'
            'dK65aJU%>pI@y2x_|f!_!Q<&c_~hYie)z}Pqww+5;Om2@kG~EN4-TL19|Xbt>nBfV^Vz}vlgVtpUjtocF~VII#X288c#s7*gGE#<'
            '%L%MgHiZ^I)CSXZ^<a^vi#U9Mo$?2(AW9yj^B8`}8RXLk_n1u_O~V9`)Lev<m#+uK`=Y7YXB=F^S0-=;0*wZ$90zp7@Cw22!Qlx3'
            'j)EqE^zS!C1y#-ABcR)3l!WIC@qM_4>z3J*_i#y1BjiL@Yd~<gU>{^@98ZGjl}PC<4a8Adq$~L5RQ%=SOk{AOBNLINHxeE!UQVpX'
            'Dh0j*F5n_6;;@gG6Y*6hg%XmoiC_ZP-Jp>aj^TiR2;)z_VV-mM7Fzh^=|7$pe*L*=;lbc=7hT{L>HmZz40hxWnlOJfd-`zp@X_pG'
            '{$%>~ql1G#J`5iP)A^$b5QX_4A3r<@9|upqo;(Qxq8qdQ{ow1_<m<!WQMf;U_~>)f1V)DkJc<AI_}iPsviNV0U;JZffgIE)lL2c|'
            'tvmB^hre-5TVfCYas<rW$xGns-krQTK9$_v_s75deDdxEa(CV8D6&MtTnSuMOS<Cd8HgKnso~>7-Ff)OjlA{bH@yh2Glqe$XVq0m'
            'f6^_CzcP9I5{wAeJe!BeqIP<nP6s$mVDs}d#uFOw#SUXwg<a&H@ryJ5dnocEgE@EsP$~dLyZ62r<G=JfuP6H7fVea*hRkxK<VV0k'
            'XqkoZ=~a|OIg&Z~F-@yT9?+zcK8a$JLJ|~e5TYIxd=5V9juN}@2zvu~9j&{)L7d)%S+_^sqqza$4{A0;jSSX%kd;J~h-PMeXz><V'
            '0PGC%6lHjvWoc%!MEK8q@7o9@)>R0*sV6U2_EvxM583va*lXy#3a_Uq;n8Y=><6845ShYRw|RKItqrB8|B|~s^HJ=Cy9gFie;zPI'
            'Z2_+*^F$$V-PL@PGy@q22>om<kwET+i4wFCp1t_^58QFnCGB1Ut?Xs$UWFSZ>m+8Y->?C9oGaKfqj7z71~-TQyg~M_Q(qSVBM0Yf'
            'ZGF9Ja)mGEo*cvQRzlfU{G`2pZ9tXK2b++jy!>McbWAWkc>+L;Ow4QBIRiklstq+;@h6Q&atuzJQi;7eu>kzTuW<?R3J-b<{+%o6'
            '7jpTxlGw;QkZzFiyo}>6l3Ah~fJ3}WvuFXMhc%J9i2?VKXmkTN2=Gl|LOqedO@QtIueA)2kXH+hXf$1x$ra*71jMHs2dl{}7|NCd'
            '<es`u9zTBgr00tXpsT$}Jd8oPM)gIPAoa{Gknfk_d;Zl`1Qo6kRJ&O8bd1+Wmw&?Az((`>0VNFpUB<I0s}s*3f-9t)nnnT$5nKmR'
            'OgtZHhT@H(&4O9xVjU$3oUqgHj{2}@gAM`vmM=NMcNwf9@SLv_kppW3Y$QP^)fuY1Lb`>*_rSQpm4r%?AQpe28=7Ff#EXzLQX<O5'
            'EWJrk%Y|yFWD1XxA|(JXGPp$h$mPN-sVk$o0=LVQZzflhUZL^@m0;xzHDa){A?n%!93!gs;9(9sKPBZFd;#v0Xg&`!RPo`u4Ah#@'
            'f7ObOgt_A98h0<KmEvNcnS3!=ghkik?exuo_knskH0UXxNA4wg1|l-dndP{()uTKufm27eBHzFXT@hZOP6%OXRDcnoGOSzShW&o9'
            'x`Kbh0l@j^BVt`)$KYxmr&shRcbEZMMhR*B0AvOJfu0Bx`1k8@iUfoYTD2bM7T<c|-lYckL6p;yby>SwKjubTeJzO5$o=*m3yb%R'
            '3CUSxx`y<cS<6i)B*JR}->VgedHx2UA7^P=s6`+1sK>p5Bx1ls@Jh@7P*24+;=p*pDnJja^=sx-FVqT}Q0`npv+@1bkoC3$)PWyb'
            '-F<WKs!Pz}g<I#v=VUkM>>~qUUawB`YPvSrZ5*@g<6Agv4W=fV_?2kVtH!|yf%mNftr3H0V-v?C)zcu&;AJE)*N%P-rx@M?pY9$p'
            'Z*Z$dvmGcpS+NC0wy|eUH=F&H0sP5#opm5lu)1JvdRi8vM~~?O1aw`42;lG<)({6|RTxh;MVNP0d%g&bMUop<$3P9ILcc%kKtG#3'
            'epe<qU}liLD1lYN)!}PRURQ9PEV2^r=^&59BtV5=9)0lap;fcVLQ?&A8<DwjjqknArVMK=_2K><OHG{YCt~UScPD&5MGez2{RgQ#'
            'j7h<UclBYdGq{+$bMMDrj^4c*@|wp{9wfQAf|G{Jt#$-07a(64;I_^NREJ3M2V;fa`ZyWvZ^Kw?V+3@owv}$O+$9Y5)|Y4iyz2Hk'
            'cOY7WFm)EL0$PC*IKdB576-5u!7LKpTl<*N4qa?}uo}1ug9g!T?`4o(#SwuEL>}gVu3`~Rfiv0!>nKLJ14kB9!tS)wX+3F;*X6FW'
            ')-d(CXUr6(40Xt^;7JCYK3Yg|uxiVVkE$*Iytgp_*gA@b`5cYH;|YMvqEPQUX%F!0x#(6+K-QYWE=@xGqc<@B0s(F#CLys}nqf;}'
            'mg0Z^&;P|sy$Tj#l*%u%^)+8cYf{w8hct|h&%GJA4+9s7xbOw!Ktm!S9hQa#GjghXl;j1l<<__W8K&uKMW4gL4a9S@;O1ggf}m68'
            'p<+g$1MWotn=K8}b7+|hQWgW-3k(SI?BY^>tsh<XEN&Stv2ulmSVDDs3~Ynfx)gb>*=y2~9o2H9Y|j@m4F{NGdIb^}NLO}Oq;P64'
            '!}p;)n&uelu;`tQYAsu>WUCy+D&{bNb<Xg_pk!AUB4L)t)E0J)tmnK-#IB)a7_KJa4DCd?V-W~F6$UWOMHZ}=2v4nM7ijoE>)!Lx'
            'pAgnJfP&A{1gKuYyOhTPnfYO};4?QtHWTU*a&<iZoB#&mXP~+14G2*rE3wL1pawh<-RDD`&+}h@9rpB=8QK<R0@VfR{rW3jB@ZfT'
            '1@4zFsXxRw;($Jlf1Mv(Aj9Yl4W*X^O0V}X#8(1<4L$fTJvkg+n7f0|<W@||C?<MeL?DQcjaBQS6iIZzhm$fZmO~T^eX$Iq#j=op'
            'OyV*d=h0#n&;uBOshV&3=R=yV#;}AS=ZiVqH`wi9un(A$PRo4!zIwVpczh=`?zFKxi7$N^&lrT^Fjkt)z5U_!-ksopKXAIuGq^!!'
            'clV@TV{%D&uRHJ65c*w600q*wYrxqQ--f}8g>FEn_uolhkb&SW@Zb(dGsZMa*B%^MRvqKgXK;rgzAiGPke<{ND1sY^=O;hBeSLg('
            '44(+#;E;C>l0>-VP(v8@Ne2PrS>QX^!b0G6nUf9zm;uJa1pQJF`J{^qO*9_>;=?f{WjAZR@~Z#kIV|?YCH1;I=M}uT<Rz3oIywA+'
            '{R#p1HZ1nyIcBdIc|%Gc8f-?cJo7Rke-6|s@!^Yaf=w>i(<7XTeqSKpMYI8b$OXAnsBr0vOTEEzbmq2Q@<8DJnFgrLpQgza77lMn'
            '7~30`-($RLfDqyaGJQ$HaF&ZV=%xf$pLSH5E(4^0L@{WV^eI+k=_$aT!);e45wNfWaarG07}E;3<K?TLj^)zxCXy~9Pqix%gTuv='
            '4oSEbGuoiq7T%;~3^WbMc#NbfM%oGMdO2B)^K2?N>T)js!sl|e=8MaLyJj%eU`?)CGt%nd#R)uut<Awh4+jFSweBIF^;Ph`%ZDD1'
            'ZMUKccqhS&o@ADTWV(ba;=Sy9p!)7teI?m&dTj3Ou~9wf<4pRrg@?IJpN2FlGQhlUbo1pcHXRNg&9@H@dtV+7<%^~JV$YPBY3kqL'
            'r1tj@NGug{QKUu)&NX~@zK1_Xw?u5W!&@e_+aY|npWgJoyj4p<pMl5Ae!8q4Z|E_@&F8pY<}Rrlg;axhd6+b%rmwLX?)l#NtvW&5'
            '3to}hDB;LW!*QBL3t+uski*xcaX{X36eUyCK!^UG1NYWg`9aLa9?u891nw{2Q){QThE~q=wzoLw?fI=f$=i!P#~5JFw1H4o6aGSc'
            'n*tS^p;H>#%K??MP)x8D;5pCr=VAYFI170WmwdXWXr=>pJt2#}0gl$iuZY|-$X38;<V2GKf!rE&1A>MQWraX(GWWs8HmFHNJ}@Gj'
            'Es(R>0Ilo6g$?H$s0kqQ4U%|}duGfc0sQ>0B@Q1b!%zeUB>hz>^wGTNu=1x5{&={LL@!0!{qWJjLqY7XVyeW!{?mN}9MSWE;<RQd'
            'bV7m@qQD}^BnCtXXfb|JDSmOxF%s(_F(9Snrtl#odr`={BLcjjl5jzKDAZo(VPUK&tyr1?nZAmUqRq{b1rh9as0lc>RW6=H?Dq0<'
            'Iz@85H~1)|hwHK!9iB6NyoK|%y*Qs6e>`(h+iLnb&rBnc@E3^XBjUtHw-r~mg<-S}aQ^!Ss1C%gGrfh6w6Bj|M*bl&WYqn+-$3oY'
            '>$lK4bHjE}gqn4CzBj3ixe`a1-@3*B{2|+l?j*bd^bM1tFvulyy*5SZ?e#2%Iw)r-QW@Md-1o*p;_D%hnKbqVur7eeAZ{hdjm_la'
            '-P|Vj;eC*NuP?~)<~ohbm13*!fB_ORK}eil$U9F^UF$H<!QQ<A+FcLs?po)%19((to_~nm18soOiLP2qV#oxoD3U-B#?d>R)Ps6J'
            '%v&SFCOK>73(uAz!HQ59NOq_xS-^l?-7EnsF6)~H+|O&`!4%UAM3k8&;CO=z<dx13@P}l5aT@p(_%w&d%>~kb<Wl<)I=Q3#)sbrf'
            'LwpAFL@f}x*552sKwe`yxn$|yfGnKn6~k`xTvC0~fO;<N!^7ij-}ZvhVsz^wO4HSgh9n1%@20OB%c^e~Sd6p_el?Jej~OaFX$B%+'
            '1d6mLsFj(87x-fpy>Fyg*g-eMe-Vq1K7kor@gwx0&(Vc_1ia?sA=Am*>a1>uK~VtB7{R+RM&dboxW9jn4M04qe^9<qBMRVOBii?T'
            '1Pn>ZGabcgf|l#y3}_SX3wiiP^y(0x?<9)u&V16Y{O%pG)|NKqGtE?cN0o_GY8gsMsoMZPEFYp8y*P)zFNpksK)IkDwH-~At|cu{'
            'SuRI4&cmrvKKu_>TaM^IOuTfM#o#%hNgNj8>@7d2Gy)}Xk&>Ary4Bx^{t~bYE|?^NYpEZj-ToS6M`p3UqCpSDJG|3!OJ7`hS0L+y'
            '-U}rL<KzWgZy9i#D6fD-JL@5zmZVAlLzoe1QZ(^rBz)hVoP8%Q)hb+yZUEx$Bo8yVq8a*N3fErI?Xj8jvc4pj$Vd~!1#oes1|*#$'
            'ktuQ6uYfD{M@}&`EH1;D0%@`ja8h7z`+3@*2N^VxSi2^-DG@A?0Vhv0ag`ukDc-7q41mPLwxECt)HW%RmagF3!J3dVL~4Hva`6S?'
            'yn>|}s;wZM2{7>K4O}X~dP>+VyHjLO5lZnM2S7_;5O7r9Etsw6u}wncapk_mv086fP3QoZK8g+aQeSM+u>&XyhXt|v7v>FnxD4PZ'
            '%=lhG-GZ7)2+YG3!H4bzUT@3bI?UM<1pQ2MIgZ**p*94+LRbulhD%f5T71*+^iLzPUlIN33LV7GO+~QRr|23Jwdrq#-S`I~IQ;k$'
            'G5n%aS-#RK>P)NC1c+>3DDXjFA^e!PK>Admx^rZE29rq(kqnl2<dwy|{}et-AJ0dgRLI(vZ~X9p2__;Z+Hv-WcNwnco-#-%GpOj|'
            'VH39wII>UCGd`;4eYhF+hZm;RPEu=ZC0$K^*BYB+_w;$h1{ke3JfMC_ipL<gcZ4<{H5Q~ysiOi9d5yMMd(+;MdA4J18O#@LvSzDS'
            'Z;yRRvTDyQjkj`|xpOChqb=tsXpRng(i>lrMD%__Ii46l#YV|kxS=G+4*89UqOg-<ypiC-)w<aDB%TK(h9aj2ND3&KkOmJ%nU;$s'
            '*%VL`#iNc3O$1~HUyD;T4@nKc7jj*QCIZogOD(+_NCzS&>SGv<ad9CgTy&(z&|HjZR;Ky2W=1yw4OoNMrA?DKD2>+A4Ybu1JjBr^'
            '>0g7P`1bYj{{DgW5#&$hsA!op^5>F~hJ!Qu)n#SKh{p^P5&1ovE>PSR#ua3Rv`JahLFEc@%26Wp6kz$B2`@unsBi@EHDJZEuEKis'
            'K*ZBKj^<<7Fbu;8XMi<ykGnQ*38rq?mM<K8_EiXYG4{nZ%ATl}!#G&w*t~`b5eL)tMqcep&{cUbcRk{GfU=BBI;8r~Rqizy+0k(}'
            'O3LtVPl{CpQf;n}GYzxRQK1r^+do%5`9j^JtiJZ%JUH^qJ8&MM3Wrol7PK!|MK7mD+GQRg^8&aQO7RrtYrNwS8tS^A(Ix5tgC)!X'
            ')3a<t?Q%4|lu1-bb1>=nWvmSR#9!oYIIz7)nX5x}GEqBUE!qQdFa|n}{~PSvi>6RZa!RBo=*~!Kf6(&<|1F{6102-I%M)*>sJsG*'
            '!6F>ZJsE|4iy()C{rUF6EkUE-B2UCsI*Z|y)h-aay+-nw+ys$2P;!^Q1zx{8+tm$B&Dr}OU1@vmbj5(IJZ>J=$cI7K&$!X?`!z=T'
            'Ho>|Ag6W*9+VJ%aEZssAB+l;mnJNsXF`$#tPtlj(QruxV=ErJB<xMfN2qs!tkro(_l!Cg7QfOfoK8jIrc!Xy5pb^;&<oZZ#t7BP('
            'TD#Fq@+dpRo-W<`)vVCPP~G-QUQ;duEE?op!-YUfZQ57B2~=UrCwf?fXszOY0(s~g1x@}n$NRgu6#wVx$s4*qIP#thsCBst_tWvj'
            'm<Msb5i*+qaop_w{J;O_|I#WR)P%E07NFa=4iH3#5akZS8{bD6d!~IuV2Eg;Lmjr|Q6ep}d4Zx*I;RhjH#Hg@RgQy!qX06OZx<3O'
            'Uk>16;*#B|1{H$sV!+RhPaxZPZSwM-wfxls-U0eMo#THnZvo+g^o+Qukn;}4{yr>V$1)G?X6IW@s9YU&L|vBs&^nXyDBz?9ltQAb'
            '8p9+Mrt_*@M}cy>QJ7IzR5YXfw;;Fn!}%P&fCxu~9OS0K&q%Pl%cVSSKH<G|hsrDaF^Y`pl4XY(7@*@Q#Fi-ZoC@JP0UF)a@{sS9'
            'GQHQo&8pc{iOJH2kHjvoG!l4-3MsoQcR2z`IXRGJMlto|OdwK`E+ALNpMf|*4X&2>;dhznAX^G4=q9YR^<GG+W#}>ch=3FAks-5_'
            ';27f`<3wKbvB2HHoF{7-L>y792k>?)G`*`8ArfPIo8EfrI1I)4t%oQ$9T%H5ySbxV2#5O3$X72)ivJ;F57;zgJlpzWFc_$#;>kk='
            'sOc$@jJ^4|@m4CnBa-vUFuBgq(=lY%GZ>S0dy{)lgLuH-Rg@L+Aqy8F+m*=`a#^l|qzYzcw<W|8_)e9lL#JO^26><6`2c^BR{B9l'
            'hgDJh(gQUCW~`<10}vZn)Cl8wWOf-~qnp&;#sRv6=<&s?Y2LGnJo7rt0`>z9R?#NS?oSd;vD_8NqZUH<g-WkO&b~P@)+{%zKxV-#'
            'CwON_0Zshy{ZBHj%$%H{$<#{Rt^rVay9xGr6Xax>zzK=h%U}`~`7;!Kl^fgTr6#MFmm)>4#2dv`IdSY96)uk6oW2r7MAtweM3+4C'
            '-UoQ?i|P8{@bM$o65;`8an=CFvLH#xvnXT-O;&u-sYS&)ri($8LR*OsI2&Bk4ar;?$=wD#{Lm^uLj(X?0yY=Rjr6mMQ8mWL4iSH#'
            'R$4DCVCs)&FZ)lQ8Sv!wf$BoS<X~=Fm+3%cY&2p&prh00uU^3zG7tsx!Qbcx(>$qh0kNPig>9vdp{PTO4;|!TkWH7}tfG3q=3jPy'
            'oqg3CR{!q)dHCx9|N3*!V9O9BEn*|QQzn;RK(`_M@BF|2np}MO=by0skpFN%v3d0VppM9BRRdL7o~+3#1t!JR8G|;Tn<J3V@5i6J'
            'q#?~W!6p*vRk<PJVX){2j}<#0w~X0O($mP18^bg0WqIS;bpQOPKm0oD!7kk%`j5B0_xe}&&-emX#(+0K%k;XtZv-8%v&RUpj;{WQ'
            'Y<JM>vCN{rO~MSWi|#J3fad;(clVbIpj1tF-g?rNrn>T^TY~1{j7=Qg_bb%L+xDDe!d7QJ+!|6<@PRSUa#R2$Wf8&;rI#2vJ$PMm'
            'Q_8eIhSVQdk^uI^m*U}*{eAHjt*(TQPoLm3R?Wyy_2l5STYtq$$X|XNA7Bn)yR~>w7BFgD=MK@UD1pMCa{w%AAT>|=7JI-=f-NuM'
            'x<F#AcnUJC>-bl{x?pT4dA?!mj`9P|dBNJv(=mUY0#uV(522&?5C5s`McPH32&BVjdyQr^rsseKWC>GBSv7~Czz~)Kz5C6{^t3HV'
            '6A5HViI)ukVe;JZ>6ffBT*OG0D2yOMYmOv9I2Rwa&GZJC%EcngEcm9N*C^!GbTisNzg57N8|+uh)~@-jX?ocQi?rLB(st8;O&M3O'
            '4k@)ewrS%|X~-O(y^|81UGDInDD78aIi|b2erWo=H*4tN&^B6yrAs<ywrPHi2Y;~Npw1{s!pK96;Re<tiHeQlg_Z3ykiJW}t{9=_'
            'N+u+2u`i%Uz9KVE`W*$rV^|)ygioNuA&}t`y=w3kjz~5J%=Tu$C`ED32m$AB8kgjt$gX)xIOel=6EMF%yyN)wv@Z_YR0#XE&mbFm'
            'QGA#pf6cUmg<L%Rz;mLG1bKbDiDhaMDJSBg!+2Hn?$qa29Z%;fr&9`;N}MR`rjayNhE+Z$4_ZYNDWy`<4H!42xZ@>LOAZa_sV%AS'
            '`g0Ni%pACt<{Mh(>ai#Bm6uk=p>HLLq#K=Lp6Oh448Z3e#iDT}o`)79J4CtO6y$GWs2dv7eCQEc1iP4<g;GU{Z^VAbl)D<gZ5W2>'
            'waUnFYo~WZS0(0Qc*%9XKz_v+zVph4V5Gm8rWxwP3>e1oOY&$EhsHZzV<o@Vp>KQjsCK!#6%cu#hH{pEb!LWYTV>F?MVS$s^)1Fs'
            'eZ!4xQ7%aaon<|Dr-A#3@<qjj@F>n5oWgeUcjsR5o;$uri&1M>+uGett*vfv+OXa2jqBTAZ|kfN>o6}Qy9O)pd9k392@nN1ytvYn'
            'o?n=fmmxo}H#tf2>M0z)ockO6E+rO_jP->fd3`b{=AJJ)cWx|3u35wG5uLbmm$9EU#-2rAERYp4-uMEkp!N2A|AHsM;zZAHc`c=j'
            'UL6A}+@hh)z1y4Z``e|F6zC2t?A}<39a#KAGP<;$^9XtkNek|U<(4303NiClVIh{*x8GJG6tv-&T1eRCF<^W}2GXpG>R5RfH7j5D'
            '3~+}yYTqy)_gbOqvFlARIVPI4deFK1WNW|?2PkOi-{sWMcS$#f-!7{r^Ni3r2e<*|@c_1{d)=ecbB(7*-fLt&&sC^@oksAFdlQMy'
            '7oz|9{sCyGN*TBalTgmMdykaeig#KCn8B{|g?2XcKyv!=&G_B%n-|CL#2QX@2t3{A9X1;U34O<T#?Ar2g{;;}B$m9i?Jr(Q=1%xm'
            'B}W?qeHLQQ!Wp@T02jPUXR4h~3_3S9#s0i%sXO~1ud9}E9OX(|LiX1#b?Va-{MkeK{$PK9-^LaoG?iXpxPLg?4sT7Rfboc+(BJ+o'
            'e{i_}5Yv{K!3a}8$T8~Z0hKAxGw-S&(tijU%?p-LDEFoc=6!TXD_XVDqDckd9Q6-D26V+kM#A?{r|C(ngy_x*U-}>-<ZFt;*?^!t'
            '5U1#MErU3djga63dHBjGcs&~S-d~k*5v@s!y$aD?4P^tsUX)c}f?5>DKPnv?U2YW7FZ7VzplNt<&ZE9iemSP&A2~r{hQ!WJ{~9Y?'
            'Thb6-OuHYBRqLjB<%T>2{^J5Ir^MycLt@S_jU<k5NCVmSc8`S8s>)#S0lBvu69S-PgW%%HsJ;WZX^#X1SChpolPjP9Gmgo(?;P9M'
            '4aU@2Evtmov3;j`TlwpH(a<rWV@xV9SD5KZHr=;;y;P<^nyTyM1%ps4jPHB(*EIUCnA@Nx;wy3BE~@$coo&e_OhNsob?4bq7NE_@'
            'f&Rc;mIkjfgw2A1xu!4<Ycszj==JW62xn}S#y9<;3f(;6u~Q6Z&T}xYzULZ5bGu#0qgL@g$E9iXO{b-K_>#;bXh|nIWe?meU3_R0'
            'MXm0vYy@l<wrET=d+#GeM9MIMz0D}!U`ba6j&pQia6JS-O^fuvv-gIc-g-UXyKTJr0JcVj-WIOgZ^Vv<v@pHxUdu<Q4U@o@7Rm#('
            '(lp3KN7$?D{hG*?rcS-OIZHV_sQmMjkWzo>OdQ=RP=xt^*2yUyjgzC+xU0r%jLR#->dv|=Q-i#V#u(-olMX`zR+S&(AX${bLcXxV'
            'I_`oVB~gLNlsL7%Ipk$pFukf$?dtpVOfO3=npT4Kc+O=?^r)|QTBbyZ77508U}6Eh{wwn@oh5OS(2c7iWxG-@EmXCY>4s;Bf<}o6'
            '&~Xf~pT%_0a}wFm-9jO4);7_7txV+lmhKxWGdPU72+)Hvm)@Il#$L|4k6Fcz1N%p@4V7XXj+As7$&@WX9yB@AF^4%DDOW6CVf0qu'
            'l^m;;lxcK+=0ATx#Hez^5uU()nQyywB*Juj*U>Z_dDC@C*_Bu85)TN7KqC*<<f&&}!^Dh?F{i07#-z_2@uvpW8L>kgkWw@u2OKxb'
            'Fsm^^Ws|%PW-<u1A^rh*TXH=fQy~Z#ds^nSfg`ENP^Vy|eqdl|u%g+HoXjjX+K;;S->CW}UuxfR&wb0O3B@7?TtK%WATT6l1-ZCX'
            '2j<esjx@&_F|!KEDqzkxm6<*u?_hE<NT<^>qgZ+}6h#R;?@<J)b7_5{&nu+_$ijY<{6+%1SW^@fx=TEBzsZuO?LDmEjJg_PeL3o='
            'fftOyAr74DZeU&1`G7b>Lz!bD*oDi_)_LJp?--lpOwyc5_!X8ALC0t~sI#&jrE;g}(Lln<sAGMvPvi{Sd-qdIx+(JP(J!M9ft#05'
            'bSW=f@0m@lv~l37i~;?lIG6@k17n^odRemBx1>aCSFBX0npMH$kJJS>QiYZ;)RZ{($LqQxS3yJ>S}cF}-G;%&Pa}Yu#L~6z5t)Bn'
            'iZJ-+x!2|5HR!aHL}HU(^$h7k@`Y%X7>Oqq__4fo;84-m&ZA)j9CX#=g84W+I~fMob@zSE^CLr3wd@&jX^6O#o_f?Y%?Db-F^1p>'
            'I#OJ<R;)Vq5wP|8Bl_@nap(s<=UofaYeyC19Q`V#cEzijBh+0k++)obt=D)|JSQa}F9@bNRQEFWzBi+HXTmL+jJ}BvykB2yTw@IW'
            '$f8+DLBADe_M>$4%7bel{p96{6c8u_w4cC4R?iG4hFOSN;CNSw(V#h+cKYUznkq_%RT+q?WE=McT2QzfW4#>c4Ens}&EG_F%U+w<'
            'Cj?G1qg^yJ#kFzko%-Q!-OVOSkDAS)IATHmXT^nKbvz8t#i}>#Im-7mIW=R8BrOwB<|XA!Utr>yn=DNh2&=oNN>uHkw6^yyOd|M='
            'x~4Ll2UE^5Be<?yb8_+tOqn=~0RTjpry*Q%VGPqpM1~g2Xt5mV@p6fu(6ChR4<7gT2Tv)uS|(D1VHOCB1}YPFp_XY{(D!A=0GW{s'
            'of|_PtV=1}$qA7C`+4jLjLoXZgHOrXs!Zl_uvmn%0VTD-KiGw3mIXJtOoM}+!wgvLa1+Y(JPQnQMu3Sdc?gd&E($QDPY@|zMT*}F'
            'fJei~I251AIdj&~20@`fv8Dhz=DYN&N?#B%!wfpFPktUhKY8==)r;dd&yUAg2Lh`@9G`x7^7;k&Onm*Akc3r*=vJM=mB?^_rcwZ*'
            'SYt}Kk`NF%ZgiNzOh>Wvu1D*R(4_r@<Ph?ACf`bxRA@$W7UEvWIp3m!(|!hn4iE9|(Yv!#cFaQtA5G(#_usp}{`_UHJNWX?J^J;3'
            'c&JYv{P^b8^OG0HAV0z%FHX*mUcXji@K}C$%w!h^-lgeDhm;ja-SByK0jQj<G*LQQugS&Tn2k(uLQpV+L5Sh{Y6`|uI1cPx*+GtH'
            '2Iu96K&I%&9CksDj4zp0I2}_`trZUYnG72UT~AqZ)>b6aH<)qbD0}#UbyNhH#aLQul;>BWO6iz$M8|Qaz#vr3Y-bAPGHLA7fS!?t'
            'g{IlF>HyT-E}7)<dQ?_nL^TVyNO4*7@H3}BQX5FD6LuZoLZal-?HN^UtOYZJ?y<|=BdhEMIL7U*0dRXuL)`)*lKG#pE<m&yS0w<)'
            ')!OD=shdFE7pbzU-&LaqRD2n|$0NTCF-s~lazDR1`|jk&GZ=6wYdtK(C_}|P{E?duKPsIUkibxtQ87M8k{2mCIR4ra1G6yfS9<x%'
            'AH7ewiIh4ff7M8)W=$w22BiUfPYl)rWU)zV7U_iC>}OR#10}zj2`z!3YZa=z<b*`Ej3|YyF+x>RaddvWfq$oRpMsl!D|}$$5}lX>'
            '5n87NHq9A}G9;A@eh@d5T&!TsN-e`K{7kp5!f>6-^paZqH?uv^x{yvNtMX*(Y#*8m$VEphHEyBa^pIsZpHL?XwTtDY{QFW}eR>m?'
            'R3#5&ND`Bc!wXl7O|US$N7WKi_m7+6gVbIOZom1`4yGt{xWr%`M46+ah32qG7)oto7zz~^hH6%)MMt91<v!3xi$RDQUQdfAj=4x?'
            '^=s{oMXlQUB5%kaCtE8)ay=|zQc5A)<;K2jX_Gc;StB5LoV!{90n_&R5pk>PiAo|MDH!^n@s|t|HdGTOb%QkyLQVYoR-$Xks*ENe'
            '`&R$4(N$|84V<)hQ(1tl`4}Emm?wI~)s1W;kaOSRg;BqYvAf(c6so8UhQ5`1cBwIj<5~x^u=Og!&$z&QJ?%$tERh2fm+~AzUyuz6'
            '|I`<ZFZLTOQ!ovs*GGdOPQ9UlUg)~Jn6n28`LJKl-KDzaRT@N01Z&=+shM(shM6l5`G%e7)vTJ%M70Rq%w|<PiBa4Xk(QD)%wO&`'
            '%-*y^8IzY(kX3a=8ipF5@vy6dg8m-3Uue6k_N{g#a1*zkT7v7=GiGJ=zAp(^9H_wt$l6z!ivlF2!Z1tV?Lj5CLC!~ELDlsF-YhU~'
            'KwoJz<-PK1CFyq@e28RuD`|1)7jV}@M-cJ@op+I3-+^vn2$<#+rRrYavP)Wj&Ra?$s!6H_hv8wW%wSV2V3boze-K~vBz@+sYeZ7J'
            'Ak~cqS`t@Ff*0&q{ow-AF9vdCV=RS+a&*_I!t7xNGP&%Xd!<(qee}t`!keEeB(r4%BaHhKV8T?4fjcGBBV?6Q$#nw*)EL^iz%PA?'
            'g&(1u5u&mj$Zx_Yuk;^z81p?WD`DIhYKsNWceL?ic0j*JK271Acw`MY`5SC;CV!y>omX$}36XUxO8-5MW*^sG8Aenk#F0)dXN3x!'
            'X$haBZ>v)SC-vmbv0QOXzoycQv1W?T>kCuKfnn8@z+i@|3W9<P0%-@k$x_O>f~-3_n21+}PUfxxWfUJ1qhIhw<iUn)c4V2<At96n'
            'A5#RQB`?CP<yA5@(p`|G*HLSTO_<jsv0)y)=NM071X%u@ri%nil52Uy@wUGwfkY3#U4oLWw)Uk?)RdBB3MZ-F3QU=mqH`mPj>^(W'
            'fH`Q34SV!oB8yzGVG6Z@#*8FDz=1g?3q^$m&^|c#7;sDXmzbsHq`*2BVMY?e8`cJh{lPv*L11y&1t)};;1mF?`y&Hqvn!c0Uat+$'
            'Ax>}l)z?Xlg#{S<ybTH-D|}XgEI(*qA;sPTH*|VDkfmM7a1)nk$|K^6V;txbD!eceGZbv5xM4QHST^7Um=~-ug3X5uE(7BtXoBZs'
            'SW6-O1bZ!DI7vWpH!LJP(pUiNH#T<9RQTU2#r=)J=TOBR6j90VRlj4ifN4dwzNy<;1KHZHQ|*iO>Fe`xOJa4~wYh&1JGF$J%0wYu'
            '`84B4e0Gg*tQNqb1-3NS==9h0g=zB_ruG+*KG+}lLY!FyOJ>F7&~U#>+j#<W=Cx(-hfrpn)U}5sGzz_=Xmn}D*E|J4zJwd>QU~OL'
            ';LX~CER^mcDV0P@O3A@|!PFL&qmxk9yEk%eq7<KCEl{i(!_*ue5%tHAuK?AIsGK+GZx-AjNzF3{!QhEx!Z=viD2J;m!=JMN*LKW}'
            'ya^tzvhFc8Ho}xX%nq)(>@JkCzFZgIbt>!Z{y1U|O+z$muuJX<hN&4TW07iH5E6j8UrN=(so`a=p2}lIF~b;|h1#S+lHO20tPjZA'
            'm&LR<i1L(e!`46%m9Zc~Nm8U0JH^xoMpdL#U6C{nnw2t1P`xU0K3#^ZU@Y^q0bIkk%Tp+>L%VBSt!Jjyctb}BQ!w?JF-O+Wlwk~`'
            '$|U4iK$)(N6F#yXqhq?8P{AH)&!`4n1VxFc=rKb!#>^a;*~~0h&yA`I&0P=pvfnuED^`6@OfL)Am;#P!gx)53fF_V>h$a5eje#=M'
            'V_Fh#2SW+xP9ra&Txf<o)&KD(rDPa^)w@cQGGD<;QMx^(4l#NoJOhTV*noH^hX>>NVBSm~kkE0t=CZ;tL)!sq2P*DHdP9tW2BuI^'
            'YvW26Rr_|i;}Z^5gUsQv(9vypWv-A^O0Zr4y#(A_$tG_tP3d|uT4G!kj5eGu)i7K|8lJgn+f7A|xF#oZVFH&dBkNOa)<Xt4<mIYs'
            'ad18HfJSUT^u?pz{X`&6)@5*nI<tgDDlG<UtS%7y@%ixa{zZEybA9bs)=pH6KD3oGgI~Y9TydjzRXO5E?Pem|xv}&))@6jt)S|nR'
            'j$(EUCb11~adYI`^ZvobdG&&F7%JLK>$-1FwJy8TfyIlzn*)Z_->b{aq`W`ZnS($Ti?RhbX&yIdGP5BDFbuQibiFZLQrS=jpJVJd'
            'km!gK3}ALot{jH1Zcv~?%gdTw01RL<MYohXGp6>lXoe!NLD5|q92UWSL|T435*df2QEK1k2f3mT=NI~k92-BZ;*HHCjq;<`Cs9%l'
            'b%z;Yr|QvovZ-p7>xXo)->G>(s!ho0c2_zqO-ecUn7rZ}eY<BH5C@~_SQ@+X?tL7)e}~i0y_~=A>jK{S812?4xQYKBXK}XLNzI4$'
            '`4#K9#Pc`fq$2Es1ZXL@LE9Wz=Zf~lstO9r_*nXmOr?3VDV1!JC)NkpHMXc&qku9`$XpCq@We=VN%<RCC8CU~?DL~jsMrdd|3q%v'
            '|8my;&+n{#QS)at9?3vN#(Bp*+qNC5SM{(b6}?LD*&n>8eM0S*m<{O;5~N+nWtWD~Ld(#q+BF+o-Iz*c=cMi}7Vm~2OCi6=voqad'
            '=xHbue_CFWccwk(LrC89>{BguPWn4N-+Y8?*EB}#`==2@YW}(m(xah&g__r=^}6+@8TGaiir#O@Q1i$iR$I-_$^ZXh%Pw0yq^l(4'
            '{>F@r*SOT^+l~Br9rU*McZ?*E_T*2BByfk>5GX3u4akT|uVyFqK*Bfto1Lzf{@?9QZcD#2d;xFTRdgk;!Wh=^j_`$9`9Yk%I}>+C'
            'F34ah<>+#^k8hPY<bK}0Hr~{)+|e$1+T^Q@yUe+=>_01ZX{3HM5$oEH^BO3FINW9T%q$?ob+KxjlVrd*!u{=J103q#yEExP9X975'
            'sAV8%=xe1RP#rKwfG?cE);k=i_LWy54RSlYm7kP(2M`|p9Nd~f>5*ryC6`I>Y6jn4L*=)QzL^;Q?Uq6RfB&EV+cO^C;<K*#1brL5'
            'p84cg%pUFu9cxM+HA?(oDcrPgn`sZ^V<|<n-<-7+2@m&oWp+CpJZ*<H$->}jya?8AvbZOYtF#N%K&#{nf5CZadn(_@q>Rj$X-3Hy'
            'Ik)0^3HM}`l357=(*Um^#bmP+E1rlfW8YOg9}S*X4hcmy5qEK`$^fTI`S)Cl$+_jFLy;)x0)!eU281E)6rH0QIflGi<Px{#*k;j}'
            '4<)ek{fjC-+5!VB?m$q@YA{G!!tkBq9+CrxcR41VDWqA?>L_zcWUAN;Hb#fWZr2+irv3r{^=JIo5dZbeb1aTr1Yk^fVil&=bP^ST'
            '{KcNkJkUL05~Pv3*Jz@kp4Lubq9yGAwCYcY-rQer#Wym!%45Ysxrd>-KkmUdDlyA}>T8{d=8nxPZ?&wI(J|Mt&baXbs3PHcIUDjQ'
            'ROh4PJgldiXgVD`J|u}Dx3H2DuprL*Rv@ZP^ouWLEL#`{n=xkKeXpn!7H-}p2Iw9e-Q$mkm?sC==rIanTK(`M{4MyY6sN!|>{O0W'
            'PgbZfz^@OHKxT_58DFbshle{y)j*IN3GWUNa@nS73@i<-o1VwNeTP@+OZaE!G^pZbbMKcg@9bT|7H8<z#|n6rs%ux5I4n@Jk*~?$'
            '+q?}L5`+GXs3DF{Ptlf((kDyE<TXPwGUFEfm9HsihG}>%HepP4#0b!FfN7c$0D7^80oFbPS&J?a>#`sPDVH2%Gjo(swFF!rkkDlX'
            'Zsl~2Fi5Ik7ysZoMZGX~+tcQ0zO#VrZYu3kINKB9d=64OS5cf-x~!`p3t%3i2o^aFX2D7-@jO^RFN;3aU09_Q?jn=Y(+DhW5+5!s'
            '{?*177_7iZ|1wZ-R1rnfPk;ft>BEgu)7}fo$K#Ggu)m6-{}7;G&@5Ojkf-0IS1MQdlB>R%ZX(|KDXYX^CG+&cv?cO$%(ONX*N+DK'
            '`(GJv=iXa5<L|=!--m)yMpZ<jw{2?oi`#BvwS%u=?f7qDX~OI%nVl|!S$Z=B%rfbM_j4AkPchfiP#l`i&EMxgJ^D&equiR$OR_Hy'
            '#T%eMzIX`)CDFfFWA`?TfTV1m6aM+)lsHi1!AV&lf~r3&!Oi;?#1{4V+cX9V`V~eWO?~krvccF3;K6-;6`r6g2F>>0<$r&H|ML0t'
            '#SxY$utC$3(0ud=paZQBv{#a#FMc3u_*(sVr+)n8i({EJ#o8FTjz@&pKf_o2TTP$0K5dq!x;=Pc(=K@662q<U8Q_^B;y?bsefRZK'
            '`LloE@6-Q*TtNT3<ty5zQ@!D0i*5YJ@V`R~EL=SB|D3o?l7AT7K`SyG6D4azgxA3aC6rTEgdC&ZLoPF;$gTw?g5rxaa=$Pfh9u#%'
            '+|`*dpgi>6=RfQaX+G#ZyX>HBZrsxiog77(N|(e-YIJSkSfe?A1|9sxpNM&D0?T<)-WOw<+}QpKbvV~x+L&V?OmlM~+)w5o{muN`'
            '0e8{Pn%n6wcfhv_3n*IbHfxNjc0ED>p+#|%+R_nRZR8D7#rn#P1>C=etGof)3hpwDn~j09&FMmmH_rs}`y2!J8~V5#^$&5hjD-!r'
            'KrT<O<z+h$kl|5RXP@g7O{e9mjF|<`Lkf!(K-YrFpwq0B#M*@o=q6<+R#y7;CG6vigHKe4HY#JAx7gInwhz$J-*Mjaaf7zqlwH+<'
            'zTeT*`63)ewoZ>VAm*-1)h;xj$F}oY+qxDym`fTlvUDW+i;XK?%>whr(vRi^x0ggo8m(OC+`GN~_19_re)zXvf1Mm`x3_w@TklRF'
            '&aERsH215|vKcGd=cGsG&<{dgSMgwjXz0<tRP((TQ4psKG%>?%=IRK^c(AQKM+?H%-)n$oT;lS6^>3>|Z}%Dm*}F2XNnY_QITbuB'
            'tO={9U~BfWll-vZ{30nm8XRbB<~#Wn<C@l|K=~W#2MEDt7__0v;UjEE0(LUs#Pd4;p4J><kT-_X#L;BHEu~F~f<>vsG>gPuHt~8?'
            '8P2}4b@mf9zIEeZHJJr4F0Oa{<niN&Pta4@bIa;3RnmG+tYWU_a<B~F%P6Ye`S9S$g_8?KCe>wyGny7<3{F1A%2OQYk4a@?b_n8^'
            '`Z4vANw#rnI)z7*cfi{S>qD^b3st*A#b6j-v5HXDU&G@{w$!1(0@bp3flz>bosAkjj*L(H{4hv@c#}u@7(tjXC`Yi4HMvPTT3r_2'
            'P&=c7%M>G|6WQCyalN9lpg3lg<B@(Vt&%S@M6-(!W_QpvTwcO_-=xJ$)W7Sxoe5bGWikP!^UB%XQ^-Wpvnapf@C&Rr%vUvrE5~7w'
            'F>{wDq(lulnKe6P2Qm!VtmnzxWH8C1DHSHH?EG5gD)$rXjv*>L$RIua**~(!;ls+0+SmxZn<_Zm$!;MJe0{5$y4G7&b*p3;0_7jI'
            '5Y!y%xC&R0fo#d_9=&8MHCEPXkMI0ut)T<pj*mOdWM`@(9!$bm73&5PQ3>D;6ZNab!O#eIPQF?LgIYEX-LKD^z`3LJ-RViV7em+F'
            'X?WSwOq{rlCS$FkubA(FL><Kus#*f!-O`$DJ&e?f@*swzw}|4QxFyhk$hHF$hX+{j6z+i!<&{8&6yRioJxgKIRlD<zv3@)BC-Xgf'
            'VbSKvzqUP5+Z1%3?%wLZw^)U81B0PayCZS&<(r1B*f}y6K=;;p0^hIiIFGURygt^Bl^LMv3Lt$L@d)j0dQ}-0v$#Mz^{psG12nu?'
            'gBxklMsd+0OnN}}^G_Lsd-pgWZXj*?<Mnr+GJvLg&Zd>da2iRCdG~oeW4G+a=XS%3PujBGAjuDe9(WD4CK$JC7pTn=dvEIeK+?<&'
            'B$x~hd5biOK45f!xB`YPqMNo*C8j4B)-VvZl;G_^c03sQRhUFcBrwO62sQzT+yl=r_gDFKoCO*&{us$V+E@(b#e;`ZQ*q3Wt{xiI'
            'LNpsfH9~Y4$5pUNf-6~i)IAD7JdM5FigW3WVUVr9rg8a&dDa>xW#i-hrdOjqwaW#jTWi!ZZ5v5|C?=SEPHuR&2|GRnNCIzbYzQ?m'
            'F7<8h&7k2`l&zz973HvUwu}Vgr@fNOEZ#s}`!(9EmbQ|%>_f3@1*xFKr?1)EtDlXod0^Ed0q=+htXcy=$SdaPe>wTyMJ3Ow4f&}Y'
            'wmDWSca+?)3DSoulVG=kB;`8?pn7_@Q?xTk8P{*;W~?Wn>b6b=DDJ!<R-IhBVoa_DIM(MGP^_cj2Te6=#W6f28#$((OHiT&SXe0?'
            'hEe&T0*4jJK<CqwAKyJcM)_e+MOQNkW`e;*{Ln^0K?{2nF)8!RWnn)nlSCzzQfGzki8$32Bi)2k@K*jLp;u4hm6Nb=hB*>3pnpx-'
            '(Jl=}?b5dj8LMoaLKS}Dn0d_6-Ip~7FZBh(-@XJ9&+tE2A)=Jni6l=7!a&=*#L2n(7>mQ4tgu?CO_fHVT8KGr)H4>$a9yNN6BnKL'
            'wdCqo`&)5FGM+yu__$C=Bd-&ch`$7?d}&}@m0zG3h)auXUD8}G<@vlMFJqbBK&=zTaBu(%0GbfYP<h!^skv4mY$7}cl1FOcSu~%A'
            'ne+;ViLVG8reTB;ORUILVlksKA~}9h?ljCRDYUB?%l#3VSY?c#EbKm~LadGHQM5XtmQ}RvSas@V3<c!8E+<`RiLkx35^^pygWjxQ'
            'uJPc`-E)O!a}pIG0<oNcOm&c>Ct)-Nkx0n~n%7QaOqIIVYhxS!P-7c*4}nJj=m{4=gkG6pB7-$IK@9YM8i+<FVG{@L-L^Ny%!pC;'
            'Oynh?)m0>>AXlyE#(=Sv;?@}~31i<gP#e}Bf1|`$y5BGXD13Sb)Nd_{@G1*3AR9?I*mG_}?WCujrzH>6kVL4=Q9;tJp7d1t&?;@E'
            'NyE@!LeVBw9w2gH;G()RXNHPD5@+8XopHhuKEOu_#*L`AsVoF<!bf_sNKy=5HzN{?tjr6H{iS&MG)lC)L6s#H<`}LyIbl%f+;ni-'
            'fe^A$hJ%6o=({HEMv|aQ1<xv+q%#b3yOupU>E7<tL}osv=Hx_G?nfoSR()hSJ3CaYXQiLhL1Vihs|4<-$#tP~Y$E}6Ur|tFPq|*g'
            'omS5>1k2Pl>S#CXK-~~s^*m!|^6@M(k`L*`#`mb{q1)O~TU6DsNWBK2yTZ)WzK@Y+w}ZNe=GNEoG&0W+?9l?wb>5chNBe6t?{WP4'
            '`1#rKi?$3uq}AfX#K_%{f0NK%=3~6V+%&Mru=>of=ON~+xir!X4aASI7cn5<jm{=x9K0J&hbdD{R$P)i<am+FoWMCIDWx2pGC6Qn'
            '?ox(=>dc~M0_e6>&+>#;DQ@N%f`$)H$)|}}r6lE3k6Zy?mWtMJIblwy+{#c*%OW?F+zgZU;pm5B$!1h@Em0}0f|UxR#*5vJL)ECj'
            'uJmVz$!k_fWo}l)FjrYJjTZE^n<$Pc<1_LP+>3K0RZNC;7$rfJi$(c@5aI&)UR_n1IS)p*pvV*h^_(ifB$T=oH!Q0aTgb3_b(~@*'
            'sf|tfIJ+m;HXuxwG2ZgGQLK?n3gb6r9KV&XrK!ix$HwPNF>F$8tz-OU3hj~8Y@iE;yCkP;#1PtV{2JLpv5L|=j3sYK!F!sSa7VFA'
            'm+1W<IpJ^`ts>ZlDdxTS00N#1Wvg}biXq1g2o#8Dk5{VNZM_gDPv{w9N<N-!6o^};Jfc<?DrW+b5&08tJ$D(-J>!XYv9;%~21xf^'
            ';(m1Jnvf<sq?!y@qgxBywyt(Y=Q^M^n^Xt9ICIx)q`!V{!Ia5Qjmt(=*ESyY+%|*N75ux4nQ{QKjOQf#(m1^`5P}gOW6l;KX$DD('
            '!9aXpRJ#wH;xhOUWf;7Ji9IC7e`386O9gPZ;4g11TUXI+#caLGYlh5SCQL)e-F}%X%sKt>+p||^uaCz+fA{M2?eV+u4<|2<U!UGv'
            'I`C$gTm@66tbhluswQYRANpj3N3MM^z+`J%`d>=~IMY0Z^cbn}8WaLa%z+Mr9K`7jI+9d!M;(~rD28TrVvcL&4>cPP4X#{gQ*rYS'
            'Tc6jOmuq`2`fD-<*3?4|=0KJ?t8M0;2mKEhw})HLZ<vS0ZPq_h@6f8AHa2#zzQ(*(;yT;}BD%s;l+9z*ID?>t(vp_+luBh4>ii96'
            'oC>9Fs*X+1=RK{RjJqOixm%M_z{pXg_C?CQaIi^rQXM*r(d=4(`Sv7wRJ9zt%bGaWZLgx98F6GocQ{qMdZKD3Cx#K(q>>MWiFMd&'
            'uv$OzOtOcmCAylROL`3KW)vG_4h07$WLC8!!w<Tj=bqdVO~%mCq#w!0_SSj*!fdYff`$W2e-*5e0`Cv6hx_=WdJqiv|7^OX_NT*Z'
            'd;=We`@<)Xs4>-ApyXbdSPi)#dOXeM_}{~)+$efKp0DW5ro4W1xWBf*22*Ow9ftw_HwdO`LfV2Q?Y)MKz-#NvkVG@^W@OK|({KYx'
            '+6JGQyWy+52;f$*UZUEyFT`J@3?nnKEK(%b-yXj_d3UT5ybwbop3C$(QsK!t?^sEDgRYk<)g@`ixwJ|E_@#MW$t%m$q6>`eMGe?9'
            'O8%mt&O$6*pD9N93+|p5>00H<tE7o(ie(9KAX1km6AjFxEH60C{-A;9R6-m27wK#;sFaDN=2R0F=j|ZKGj(FS>YdM(kyUtW(}Ao4'
            'i@5K(nPKL_B(c?LoA2E=w%)>p{2?x{aJP`xq(w!iphX<4&}^RtRNn~Ab5b*A1v|F|57$z@RHn<y_H%ENSkH*pE}B<u8sd$2;BwVZ'
            '2OZb$s9N-K)AsnCS_%oh^y-OsA!pWj7}2>85B^yd6_5GVOzNm|-s!OT9hsb))70|LjZ{MM9g}v2AmYF+X4!9*lVqBpkt1=L+=`u0'
            'sOv)L{r(MY$i6mmv|7gLs`}MZF5lX@oElHmE>^(K{A6{PPw&&cN)zQkP%m}^Q8+lx%QYuwmr40#P6sR5jy|n}5<|VNRFzm;oRR4a'
            'PELs<GqM;2xmm_zKue?(ub-QF?=GtJ=kD_owr}Kdq*PP1-3qQh#9SnD%`mny&dL{iyCN3N?u{AY0MM+=fm$xg>k!ssZ+EK23Yw9M'
            '?}Bw1p30_3!Csb<U3GIjN6CwqE(Ugol@?+)j#6rKVK-^Dks`ywg`t37c>eAs5Ce>A68O<H<Furxbdkn6rcpnmqUy5Y3zw=U89Q?x'
            '|NGJNv)8}K5-^xhE&||Gra-2(W-Pl&W%=x=(6uvguQ|MMCJT^pidvGOsk#_tyY*2D3^Dh8rBcIM1U%kcOg2T>he`DD2jBQnQ7N)G'
            '_QEjsqVT=KOjRoLr!t4WV|ChTLHbkFa;B(1%PBR{O#HPBC{qeAW!_Plj4#c-xg=wyz(a!<E9_GkBtvD*3rKZ_Wr2zmO_-$kk_8Wz'
            'Q8x3XGEMBwGGLANs<LDeSu)}cDfXyJ9<I0=5+)gvHbUkW&;X)YEosNbs$0u6nwo3DD=d&U#F4adB;{%nVwQ?YSuig~POn_nP+)m6'
            'w78OXmt13IrRGbiB~TK^|Mlb1>sM#LjNiUHK0Q7=y~G6IY*69ae$wWOUdRb^Z9%puS1K?@er}SbH#uUZV?wbpvu4(y6ufpDj46?S'
            '6W@N(AjV<Tn%fKc`KGpP&vaYU(Sr8Y5$}3s4401-5ioVc{@`(swVxI0Fs0~vIS?agcJLJbht*aq%Zk$Q%bJV&HdLnjPq(s|?TK4!'
            'zv7is32io7Wz=u&w}C?SB*s|e85Oi$P+k;+mUB6i@l7-<mR88Rdb14CJYdD9cdQ%?7A2I$YZ?h+%_ifxut?jBUdp))Hq{~P`;A<v'
            '9oPQHegA&eqfaql)%)CK$Wpq3oC8QzBO{{#$AdR?X|ihRmrizomb`~n>uM?QXx$vRe@CaLX#G37I5`IF)>bPc>%g5bO>}TT);M)O'
            'a&0j0XE~%I8siUXx~eh;0Ie{hhpE9lotBgpqRJ}p*tgzo?03c?+i8n@SNWkR$w0}5rhN=cP@cSd^_N#~j$TWvBzZ8YR8tKmNs#~3'
            'gjfj7P}EnHjgnn3r=pSr%w&KTF|9e)BjJb+^n9XPig0@Hc6t5!2fQE8&Q7(>iXD*Lk+XMKj;20V0{BW_n9xk9S_Vxb$_)XSQt4}j'
            'vgsE^83{31^EwoUol|orq%EVNIHp_J=n&*_&e9~KL2Iu+t8ML{!MMiT@ku7Oz5AQk_Wp~Q*c^|ePqw$!e3yO*uki$7&p60%i8*n!'
            'Gqm2;bvpV)lbvQnw8c(8={YSvi@okIz^6yZG|%Allz#(b-N*Rr{@u-X<{{kaXTHmA_-;Szrd4Q!)pH_coY!uFPiv*Vs(G+`;@tA$'
            'qfi6s%9qT@K3I7?4<639eb4nuuNsK29A{|kkY7ktX*WZ4Of%9)+M_<k@^p`0vQCBFLt3mXmJu_?fQ<8L<|_+aWe98ffNSac=Q1BZ'
            '6E3=N9S^Mkt;l4{SbJNq=A)_1r*(_y=deaMKy5cmyPy7B*r%JOS^vIiv2K)_#@dMC#5ifk@1|km+SEVBx;?emxHe;Mgm*>n<E-Yj'
            'fi{`Xo7P|li2JO9r(Eg1`<UNt=i+<Y;wNxDtHa{h<!cN?H_ejpYKU2n@xCqKa>b%EaAPK;+u9f5Sh<YE4)eD|Wj@9XZta5>g$+7Y'
            'J;PiB_J__r{noo^07cYlMEa+i{M-(X%G6vqLo+35V|1>7-3Zlf<c_rxxDCj*6y6BeZsJ1KGP$z?)$=Bp`fY6mwDjHpkDs@}qqhz4'
            'SiY};laHIB@yi;pDtF9CR_SR%$tt&o&o0_M2aKIZLx)#10&`Mv_>^4GzI6$mFsVNtlrYK5%(=m-MYrinHg$pM4ni6<W$Lt4u3AAi'
            '&F1<y+agA7c8^B2pR4J!_<YhT%Dl{noYd6RH;ON<FZaZQ2jUPO9$G+H0oe#ezG?^4sS-(=PBYdb>DT}AmSJz1{ky$oz7VH?#$k%('
            '+a3=Nl-ec>`@9oWSm@~Jxxf<1xyYmUl)N&58;wjEuTM^oPt7`RMK0oCgSHe5L}gJ(RidRdT?mX}>naq>fPW=a*E%=Ak;cpWG0}Eb'
            '^>y`;y(fA+d)`1Cp`%4XsxEd@*HLb=xFcq97FF45)H2chwW^FgYE*G>tA;Lr(yX6EmNl&SDwRvh=#l}05=^jGHydOIP2Ne?);eBF'
            'Hm3hBS0#OLu)c~f_+~*10!NUsJdH)@f^LhX?K=lgW?c_*$96&?{+q7Uq;Vh%EXS0K3ic$c3k^N4;K6GKTd#+w3bW!1JZk64<|89?'
            '>fdy^D!WXAI`J|z48>54Bg8m@tBA9NKGOkYeR?kI@KHn<d1eKx;KM+XUMpKaa=Rbve$$`|(VAOCffJ{4p{iOS$E97wjuLV7_U*fq'
            'pN?Nx;UZB&T3b7AgmjRkS4KvP;wn_kgak(~NUZX82!{-`;=;1gds)kvd54x7(Y`8POjS+$TA!;8luqa+7E;BqmoyK}n$IFYYpyhb'
            'GPE^3K%~*Zzl!nYa;;dhSqFvbI$OrVjOsQ6M5>0EyIQ+mRU}V&YEY@=u`VNsihlc5<vF3>Mf0YCRrm{-r5QG=>)Ft{7C2^B97X`L'
            '%*XEj99`yiZyq~VAhbjYVy`G6Fu~HjxSWVaWmW7|<_Xl3YxthIjLn~e7Wewvl+z5Uh3M652+?>N2i6fSm2~bg*9*7I$~lgtVMJ}!'
            'dhVX!MP2HwMDF>X+Uvf7S+}hSVCKW}%H%468_e78^;+B>Pm*8*BBfDaMpZ*W?n!iSF_j(f?p<C&^F3<Zj+#b2NQI$WQWnO=D(vl9'
            'wP@-X<QAo|oHy&uw?OxbD3g_7F)?-)VdStRwlj=li{P>0xoJ!Cwl`pHWA5$HS!Zr*g<<aFt@(Q#{F=QsYnp+bwa<3Da;l_aafR39'
            'KZpDb#bD4`Zy*b*AbNGCt7DVvw{3uTa|rHztL}e1xVjeaEE?#G_dmicwG5R^TR0T~XS`$aVj5&<j6mNcc3Wesq?Hi@F5Uv;43*ba'
            '#5KVQW=w8ObQZAq9J8{5Oo5WOg{BksS?Z{?xk)sKdHC#?9V9Xk(g?<;>=RU+kP`EuIX8!^HBYSs7OE6F>>w@~#dFilkZI4wo?}{l'
            '`})UIGSV7m=so%}@y*L1o=b=XNmejL?d~<&Z28tsQt93|Ho|grxhcKPHV1as=67trkRVG~u-BJ4VUqjdVS`JF?&>Hwzr9a&XWp-)'
            '*Jr;R{l}}*SO1}_P}kBmodU*O2f*h5m^=ltC!%SX0RsrEs_3rD57E_w;0nV964<z9f&p$SGhD9*?TE2i>I#)aR3O<#iYE=Y$M4s1'
            'nuXb~-Yn2_z|X>YZ~!705b8F_g_&irOj7HvS<l)8*qy9smftHE257<`<|M7~XV<2&6_j4xIhHHAP+er>C%eEZK6<AQtaq#XZ8b%p'
            's{+$tICOs7yQ5k-;%6PlcJ!$^w^Ak~vpzUt73Bw@5N|2*ju%<5UTSZ4{g~@u(#vkg&IBwKKc<q$x3x9CK!+IC$BZfu8N(@t-C;(S'
            'bBr$^Z>+}~7ZWhi?`}*`cjf>!oa~BXU!p4Ur+J|7z6taeswLYyr-9Bc65wpkuR+Ye*sHgmQx1Jp?_(O&YTMPAh;!C8I8>#Q%3@*p'
            'kg+&i_aBpqeysXXEt77pYSXEeNc0=l+&%zRA&Zfz*E7oCY(|!<&hOL;)u0}JR9EKvRT-O6T|aNbQ3~EsO-*c;6RGRCT_~!?NL_nv'
            'w{%tWqeg|l-)XOAGip%d+wHa1c~n<dJM5RGQXaLar5h`!mJ{ApnFAfKPX5JDk^5va(5+)auVZdwF1<eLN^IuPbsMjt-#Bf{VmG8V'
            '&}8J@wv0?e>219|Jjc~=m?C=3Gq+FjTpwlJ9q+cbG+Xc19q)FSFM~C%IU3sTar`wk+IjlX_oXQh-luGqecfp=l9JS1F0j1IDW167'
            'niKBUw<^AjsxnNzRqADA)G~1`>_&N((L>d3bm+@cA;wqL`Q3ix=INO?Zy}%)XT`!1$QJa+*1N~{Y4+lt#G7|xjMc1TtQ&yk<)mMU'
            'efj#%lrEIokQA02p4oikaSJLJRkL15Jt1pnDZcbYbQNMHr&$*KCce_lK@nb|8y;|YA*Jgw%tXMYNYRNM5s_QRXiG}8UnL-;+V)sa'
            'fzrF-zq<D}jzlfGYj|ZZAWp%0&|nbZU94F|?rjipY-GwNVni^&gWksk6St}BVSwrNJPsD5zx@i0FkFs>zgFA&->uA-h=bV=lj%lg'
            '<~BD><rSmNY>JMQ{-o0P?s;@o5bh+=l?qpFaKn?It@@&$s&Y)_mlYLiwItM*V>kC%`L_ZQu@(kBvunjdv8b4BY(wjzFAg4K73PlV'
            '=i^M~TrCPMns<S2Fz^ShYLeb`@y7>X`UYh&#dN|cQ<H8hI9TeU%n6Vut*p}jcIL59P_4_F3c%~=HnfGopSwRWmfmC+)+OM{-2@q;'
            '$Wn?%TA*6(nM|RJdEz9^VBJ@gOhcJi551>o+f7y5U8C(-5+XUO%z|YA()x!uP`E|ket=dMIj+(uy`09g((4#Whi#NqstrU~=_s!r'
            'HD-8GN!!pLZWggsVPqESsD<F@d~U-ww`AT?X~$BneRI*LOLDU})@dx4X}D_%RlyFnFyXG-!#*bm7}O^-L!DB=jfN=>un26<^>io~'
            '<<*68|L_Dp4?@yyS(Q4ESnb`h<G3o-xmh#aF%8(-cdX7;CGl$k;AG%8Dv<?nAEP6dGrWy;I%FyMa@DQRq6e3;QfE7Fu!?)ftt4bG'
            'fo&+_)=m^G^&;BTGA47>f5HHg$FyALW4UV1_#BPG=bT4Nu^V|Jh8BML?t4X|h7On7t@3*_O#mKaRz4F&xw4fma3w`G9sL56Jp<nR'
            'j*3cnjT9|RY?>zS&=uHJZO3>)T%W>)IWg3L-=Dns@$`oy%qXo?e{w!Z=v-})r*wh7xc|tT^_C5T7R`gME&y0unKBjDG+mW3Zo#gB'
            '^lIhw?tM8}<FU-`3_^1ulKof(XoA7O5>j|o5`=+cK;}T620}2?LHNeoC;>hLG4!_M2Jl;<nclGGVQcVe?U#npxAC0|pfR*%C|WW>'
            'm<jwH*L}KBY-j$)ZiYuYey8@Z>a{hNO{YAqEnIciz`KSU$QecLHbY|JK5HHW(ac4C1j>M-mceTY;B`wYdxpxhx{e!GuF`Vq0$bIi'
            'USnga_H^3Ir@Ojr8oO-VE~cLz^swCa?3SkX4q95uyvCNyXje^})O()EqE~Cz1l3~StEP=mci1HnSMO`J^y+@t+IyQu(CW#34>+uX'
            'Q>X3vlb}d*`>@qvybVG_(iK$}?vAu8e1c4MGp0H*|88lkoKUOvYmOMHR+XN+`BG}|^g6bIldF4gXiWGtYq0~Y)p9pv>H3PclRUEE'
            'Y9C~`NgKe*HGSCN;l8Vr#Q(c=<#s1|5$)32%Bnd#Kdmse4UY00X#rAmbsvg*Xw<!Cr+2e+n{wT&xwV_UTI>T}&3E1H)L<&`Tn}}-'
            'kF-)WcQK1P-em;Poocln@3aRf_R72sNdcB!H}FP$2^2b&5~(|#*LHG`p`%r6Y>Rrfq3M&}c)3@XbnSkxQ_ChI`GxrTyI0S@GkqWN'
            'Qp`CMbCgR(2l^SHIkX@j$N^>GLAPOaYwWmXf;pqMN`o)U3kNzWJ(f3-{u*(bUgfO)8q#xBTBHK!6d8H``uOPGDY-{0uF1-zA0>)?'
            '#-w@VncM(ScV(C9Je2+=1-ko}Yw1y|TNSAw4J)*DfN{2)$~T0juH7=(*IS0?4;+j!5>r}P$7=E&$xL;xOEZVQ4PhJ<QjzG@i`qgr'
            '(^*s22dz76x}?eIqYwl~x_ex@WsT4&rzBaxm3<5BHb&O0p{QKr+~ZR7bHZa&d)U^?`bfi5r(c_xi%fjWE0-zB3RBu?uuQq1%=6|B'
            '+U{runpTgiT5XjY!sVbz_ez9|tYs2uhOVv0>DFxNfo{z&-1!)0cDjjB_E0u^3}kG!Bel(;$)s-##eD1zq(#Std)e~JfzYawLWFI5'
            'SCj0NA6Dh;K6IQG9inqoxyGSO8+@vqaqbc#sz?lws%eKgbZ}dhc{IfUgrbORS3rOX`WKi86+P?d!s~}gm@PIy%N7Zd*aQf5UQ%U('
            'n<$y3H-h$NL$M!GQM&}VEQyxliPR%aj-x9qq_hz!5p7-j)_16AooXk_5g+~cjN%DzPK=~2Fazb{7l#YDPyl^HZtXzqVkH;DEiehg'
            '8QlWPt+Yp8fe}H&YE?4ZtMuP<piSX&Sg`{=yW`1q!Op0@<Ej${h61E>7}~thRgbxlf;-<7ixN&TRe*DG)YVmhFZGA5G9o*ZS%M)y'
            'sxbysh=SY$o2$!ANt>Qwo*x<O&{wy<I~${j3<V#|fi^D99K0|CDo%sJ5(8JxPgmmxPiG~<HN2|PP%gSPHr=A(@@nep+^cmr>Sjia'
            'Epc@&j5WGO{o~<IYi46Vd8;2FYhYF(uLB`(5BKRkE&{`l*<fI4R!taoQ|v@~_xdYe9JZ{OQxvGSa)up=S?cQ1c!ETlQ&?3GgF-!Y'
            '4@^P)vD?1JcG-O`9U7ozp+0dxd!}7H9?kwL1)<7^cW1$MGbNYt_!`qBB4Lnk+C3m^iSu08?o$oA!L`k)i1_gt_S!S+wST8Q<=&1|'
            'd+p$wa`5fUz1I#3E+V|o#vzqppWSx^j#x?Z_;nhWn76J<cJ}bGFKZtn@8Z6<lBqQgj5&>}$J5<O1a~G6Z2!{zYEs=RX<Yk9+Oo;*'
            '$|83cw8r#u)pSr8xi=o%k&Ny;K-_*DUSmKY=Uv6pno3fHlbLN~=_epVlo6Jbr0zd)>BS&gZ;}adN&EX_jG!$iA*J<L1yZ4cA-LQO'
            'Wf{2BrZN+(l?zYT<T6sr`C@?#A5gZuVy3EiM<9PO2Nx)$WG)d8KP@S7K00LhD`FN*NMDIn^d1>EPEnWW;!f<~LeYmnAe7-BWM}0@'
            'k{cQMeHjqR*?sMlbz|kmz)8T6xa0h3H)JZmH<T)tja-ywVO>sQjJU+8hz+NI?V5Hl%DM%D-A%B`d#c8tRvpMM44EgKmkBeI&yU_v'
            'ZByn|1>0^%)pds4c!Yt`S2B(bSWc~JP+4X%0-JawT~`w}j#V+p+tP^Y?gs>}i@|8p#enM;>SYy%(-B38_o`?W$$u?_T-;zL7-U$P'
            'W!{7^oVS1qDc9#pjk_7zuh$`L1_h{81ruPbtQWeZVp|mwtC`9lVU-%bcc}`6%K7llpMH1LhwB-*W{t_b4wQ(AGCOlo#%WO!;-IHB'
            '4F2fN3#_jj<XnHDVF`>(!*!L<ItvZgk*+FnE9dPCeN$G?IEX+!@54$i1tryyGf`2nn0k>VwGh5kl5D_uJw<oWnJBgVbE%iB8qB45'
            'U_7rG0WV4L-~-R~Xe~b;E~|&ZQBvCC)q@8pOy*$-gaek6N?x)c1xF9PNM^M<tzU=}k^=x}I^oRAERacGquv7<N7AG0%9JEK=Qv|k'
            'y%Glj_qznAzRbli0JqZ;^Iv1VN}yEFWL|+JPvellkRXU>@Cub~sS$9c6mh_}g$d1!Jb$>XAZ7wGWHG|re3{C!bon|;5+!tfj>MJ!'
            '^f>_8-4z`GG`FrqMsilVs3m>0Vq}%K*B?x8uc1*8p%L(a=*p}+-*cf7c~?~RcIP*@p3ov5$mNSD?(Owzqt!?Da}T%z+%o2tFE<a{'
            'zJHZjf_<RdAXbe$E8w#c47$a^J3`T>2E9s<bYnqZ!c#cnFOh{-SxryFoE(cXIC!%(-vsFXQpRP1?kJd!uIU?CE4Y^ARO*;9v4uHT'
            'Ysc-js>Ey~9GUR63B1%=S9wxRJTbeYN(heq&Qf4acq?<0y5k+tCn;s=UVDO8BR4CjW&Na#tmZU+Z(`QQIXYQdjqm-flC=(gN48dD'
            'crs}#)sS9AIVZ%MM`1iemvW$}=u0`;AM76(2Ot0zY8N(9|6VPPsqxPAAq*5|A^uM4Xx3O9*{s(w$-C?a6QKJ#|D5!C;+tv)9RtDi'
            'JlXGbPIxs(oaojoru)LJi%tqBW377nE~tUunE|;8kGxrM6;V>;IKouf9|B}lX2EI^kRxLqmC$y#Df3kc_uWt>G&Y7%k(Pg@wEZuY'
            '()M53-hM~6x9_W*Pb!PYzmmQEE7{w>lD&Q2WN-J#^LBTNH@+rVwU357vg%_{r<G#OVw|hS%rt-QL~KV17MWO~1G=4i!5|`Sd4Cik'
            'W=X;H8O0tIL2i;H{Kj>-U;dvKO>pz$f3~z7ADxxspEo_m?+p)c2@U@nhK2u*kZ>Ch|7M}!e8OxT;AbWR)ajh>TFiXIQswWki`6Y*'
            'z26s1?wfP?J4cp(MbSgQE55wg|NL>~>Q$rK$GmetM*#YHz)${(BG9d36#tk=^kV;C5$OMgBhcm2Kj7`?EB`<vGCktZbkXIHj|#LF'
            'Ji~y>S8q;_-iVVoN58xhFOL86>h+6P@62#dkJPi6UM>(64GNN8in%I-1ZyB*j+)ON!+ytn{)Q3lt_uv==|;$zFM<9HWa;O7a?8jF'
            'GZpH3WijtR^?JR*GJG%Hs(a_dgC`f#DD|T<GTOF9&c^%t=<N8Nw$cr>mzDNbLb~Ktwd2vta6$%q6qqI;Ko=;DO2ZdfyiyUT$)d7Q'
            'qL>-RbGUYO^;I;rca(!=P*j<Yqk?=v^H3Iwla@)xvglY9yHfCz5UcFR>T_yedfM2ijErm#t(h9{%s!y%i&ws4yNyt7DF-UYrzTKs'
            '7NWbMAy$p0`f;q?7HuuqJuuC%G}?K$(CN2rlu~_LVL6Yu3I7`-g8%=ShQTyr?HEI)h;(x{n8et9FKI9~b0erLg7l1tBk*=>Onp8J'
            ';fS%JOXd@7NL6)L-iSK}(`|CV6a|cG72|QiF!b7ZWug)${;%FVKYo4u!|~}W@$K>VuV4Lg^3F|3+2!c<lCFRma1vNcDowJ0ryFcC'
            '&VC8U8Kik%w0d%oA|B9?QfW4&t3oGL7cFcPN7FKV1+t7c5ST<6Ml{7}dXQ)7j@n>GgEYrv%Bc9S0S5``Gi6?A7}!55hQ7&_Q^klh'
            'ylU-k`+T|(?6)5qhHuxD{09Z(SBqb_+Izz`d;ULPK!1B&{>KEq-x0O{_lnft6@UNvV(vdK&fZaApewke(%2b7YUnhe>kfs7<G$`_'
            'KQP+EdGt}yrhkVB)BDBueM}_Wf1Xf_-w{3WQK1_DB||NyF?ttTZg|=g`%T(U*V<E2dr2nJGm%Sj<0ewT)<yvGrX~8p4$$FIdxo&o'
            'iGq|buELG4_39uBqW3rXYly*MpnI^^o_nyK48yZnHROvgmFnKR2Pn=9PLO0<!f~hd{|A;Z2VD'
        ),
    },
    'video_analysis': {
        'sha256': 'bcee971d6885d7e58b40f37514f93ca63c0e10cb44904470d936a7608d6f45f0',
        'encoding': 'zlib+b85',
        'data': (
            'c-qZ<YjfK;w%~XD3XHmHEv6+!c6a*Db&q;0lQ=V}$)hVyy6>#lB~ub*aZHiAeArQPUA4b#zijRQ+VcVkfTXNsy0@mQW@3@V0dR0|'
            'aNg+m`|qNv2<J(pZsTQ?jlwicc12vM$7-FGqfNGn(o!v>GFp^zmZ~JXju)y}hk3MA^IdOpJfns)-EaoYxYan`?$Wsm)1{hBX1lB^'
            't9djFt1{b!2y3<pi!#djvvIHYCM%<PmffhyvAT((ZK2|_P}yCoHdRu_qXj^Umth&Hd6X{Jn=rqL(`)tNr5Y@<%{I*OR}tMtd6@Km'
            'UB|^X%GD-YMu{roPf;;c#Ue^0nM{_$0~}kn#leJ$diClxe39i*UO<B-bhD#>gJEx8m8x7vDqNHmd>JO!Sss_`4Gys$O`%33rw@P&'
            'eN@Xh8*QWTrnik3IArb@;U(Du!145o_!>uHVI2p9b6RCNd=et`B)*N*I?5weWW9M-u2m{nD-?iA2x{~Z#;J;M6R8DEU&SflIfR+6'
            ')FzD668?u^m(@0j7l6-UFHV=yHiG|4%ValN=JBe;FBaK0jtW?s_kc}AWd(GSB8C;T&B7%tPp-DrJc)~SRKT=$<r*6Ph-k^uWT)~7'
            'KMG}{L@B*h3{VgQcYs4^#qYLxbQ?!^YLUcSq&}QT9+g#|0-?p}XcKL+e5YXjLsgV{1uKz9<K8ovG%nb{;2}^zo6xk_W@!;4Wdr`G'
            'FPsvfcgE-~#{VdQpvz1x<KkxA>-YP;Ri16sY__Tp_-v-)%{I$R7%V_1nk{-gd6q}|ucDf7^K20ng?>`Tn}}g;!*ZR(a}DTc_{lGj'
            'j`1`7M*kjC<4;*?z^}ssLFk`2Xfuek+H7|!EL6Jf_0HcvegENn_RG^(AI{FzRDJF#_^%&*gnba>jKicqRQ)BaUy^O{!EBp^i)fuC'
            '%P7ZJyacm6`p+tc{oNljgm4S<yT#wC542RRR`DXn=C~O6a23Lm(rfr|SrOsSaOrmd;vPo!;pd;<zI%W6e0Ki!!@Fl^vtM35KYKen'
            '`;Yf$Z_Z!7ePfMwyxD$_fX18bmjAusfA6CChW-d|V}9}dEkl3t_8Cn3-PvCR(x6|IVP4`#n|QfQBK!-88~<5U1+brf(ChV<(F&;~'
            'OKzi@dF5ex9St4<tH`Q+5zUrW&b+FK7DFH;Xb%&+h9}cEuttoaj{lI7YXF;AnkXK_j0Raheh?CW5963lh8}Euc=G)uP@|tz2|F{o'
            'Si$ZrVZiv$<td{Uc|A=R_T#$Rgz0FN!!j;mn<iO$J)+&Icy_ogfZt|mjC&K{XXz3d0RlSv7#Fmc^p+#egPRb!Hg4QFmnNfmSitrT'
            '^XrPYoO%!2`WcW0@LvM<4E9t8dlWh|e5!y!sjDj++gDd=n^eGq;<N&u!Ek7?6mX_6xeIqik5LtFw$LSv-(uLjn(&Y6CvfPf<B5jU'
            '$DUy#&}no7+ZA4(y?rs(7^8pVm9z3NPBq}A?DvCHvv9OPK!1evfxBkNCntgVRIi--wY{F$0GHG>fkAKwe^L{CX+ZpGswSrn_2gk('
            'MCz9?siLzy&+<Xv8_ZA8L;)Q%d;JN4Cz~*d`*qjMhPnHp>9`d&h)F<?vnA`nH3qio&*LOcH_<Li9YCk*v;FfA`Mr#%%y$;X-A!t+'
            '*|K@<+5Be}0ii8t53B-ENT436A131oHI6<mqHU=L?{`}s{?M9AfLs{9*fZHaZ?FQ6mLf*<BiD&P=?6u)iwmGtjebSv1c20t_GjxZ'
            'e6zdiQ<d_OtFmQy9V_74sk#YwX?UZYvG5|qyDS$h$E!9pJ=CE)pJjLx--IxoFo*soM@!h@^J-V!xCCY`0wCO9x?#&Wf>z3#PL=Te'
            '=8iNgn6$N$L!#NtAB-BA(FP!)oa}{)2Tl`YH$(wdM0F#F)>}A?caAv3`V|D*zNpzJke`8#C%|k$^5<23kX(I^^|igE(KTRag;cZ&'
            'KUq9Uqk?FL8Pg{GIDqL7_aHf70K(AntV)-Ix*38AhUz=`AHS-p@H;Ir%xLVhcr}A_V-dl*w;X`ba}6Y4(D{o81J8D`jutl{Do@pQ'
            '1kz)fn-5XcYJdUp1lt<wmgY6)<iVgGN|nw;@|^L@<zTT{4wV^Deewe<2mwNzER{4-cWV^H*ICM11|F9SK8=q+naDB_8)5dqr-011'
            'Ky%@>2mn`Cpl@xel#~ZJnKwX)Y6&xf4UQ_wHr|3*|Lb30zB(hQx<eQ*oZLTWNkUppoEIg*LtOx+MSL2rKqb5j^JPII!4e*WesBYW'
            'g%je%3pm@aRkT6HFiNxPdJV#Rc%5b-kSq!~^a&m)7@(ukFvcLQpq2yr$cOhYMt_8UNaiK|Doj8iK)nM+6V}3TWGFDa{@X7EY6B}r'
            'DhHmp5o%MQQo>xqYnUNX{7uZ$3DvKlMP+vd^-J>s-PVyK5D&-$jJ`1b8IYU~>J`StE6OFP%2WHRmp`A;^C-_7&ueZGeJrQ%^9sZg'
            'P)IYx=CogxtI;2UvLf7O#k4QeZ#ztx91wuJSsPr4w;v8t4+<uQV;WmzF+ep2=P?<d1dZnN9Mxg_iSR0GcJCPgtRLzk%^`rSRyYs$'
            'Zq0@;Q+M<%+knUnC(O_M!0e&Lm_Np4A9_;`Y?0v4AmyIXpD4Q643MB~f#L@N3aT}8Ftfl9faHOXA=ZL!=HH3o?}FafVUZvr{SJct'
            'Jq7Ne?<r5hZb_mT*gs~=I5*swpP)WG1*sW1B`U)W)dcxPoNlX9B$T_jT*LSk>!iHV(14Jlbpv=i8Y6bdS%fz|g-3eFUtKl&25xl+'
            'Y6i2DqT0fj(Q>TbLbv%H9!l7M8r?<0nm_{@MVoE8qmM!52GxQG2F%Wb53>dEu*yZ0M2iyPzIUPJXg`4u&^m-Gj(XHW4K||q<Cyf^'
            '0yYw9%?f$p7N$>TDC-P^;o!8?JX&QrOO&X`HuRSUB2F!u5p+#yMFx#e9UTM(eBELU3Nwy(5VRfdFW}9atbBndsh0NtW(%M5Dyd+q'
            '!1qDg*D&w<PIsm&IGoEt0RPO;mUoEK<M);o9o+~=(65+|v<wmFt$2aw=Ab6+(R_|bfq!hA9Egt_P))PEa`93F#v!v@q3xgr4DF&<'
            '^FyEmG<DG*K%VxPfZk|O4sU|*>LoO9knHFc8G1gBlk5(K4utri9j66|yKbOg(l?hj^6Cj$0<+m}AWNgU=N}nq2l#04;5dTdLF7Bc'
            'v+CQ3CPir(xvEmVDr5B=6_Oj+sIXzHPhkpjLzv@Y=|SK34R~2JAu<#;>*)^FO|+YWXggnqgsxLH@UTX_VV1(D^a5FfUJ;mN?;CZF'
            'j4;|{c=U|!vV5u5QI#`i0SO=Z8OV3nG1`_>%fhaW%Xm<ZduOlTz6Zq3etrAyxn+$j)*yQ!`sok&9RHq2WghP8zu2sZ=ZOB7ZGj-W'
            'm23IK*b+gsBa2~ty(SACsOEqg{Hx3>{+tZV_(wR;=vaiA>E=mxOXj>apif&KLER+NBVvWRL$eV5HBYjPhmwca2n2dr=WqsVaPc}!'
            'x%VQYd1gtq;eSE)r|02qSl}Pm@u!3gj{tiS(Z{H+Xh>zWPU0IH$W2H+MgIUfVNLeYGQ`QEt&$E-5Y+N;QO?NR0K_mO7r|h)Lg{uk'
            'kJD4MaDx&O-dSvT$b>k9(@=g>v-<Zw3!*Z{pHn5aLEAcF9%^}sY?7Wn1INj;3Iyc~18Z1={njzIC_J*Wl{W)mA!VjQIaqVl8$ZDY'
            '|LgeVhwmRhefo^-QF0Cnj?i~-{KL&(KdBAqDeBj!zp&I@0s+wHEbs%@gU{1qk;f#1J!YqiIS$o3<N}2PH3;_;aR3r6pjnlojI->C'
            'Sc~E;$Q8U8O6Y5>k5YFD^wHoxW!~d1j{eep8nnP1Z^Im&XPX;j*!-i=ay?pJXW0$?Z0qI=8uyH08N^-ZAgSpPNOwdVeYA!#&gNm7'
            '^2Qn^*>!?`1R`|AX1^xCZ`4Z^;6~&_BeN=qJU~-0mKc~2E*9FG<$Quu8Afluzq%R~#nqJs>@KT7X=UY$7k@H2olI1<MI-M3!9^cK'
            '6yXCg>Z+6PwLm0p9}65ZL{dG~d{=^g_$%saBhr}Au?V~sMOLk90viQgQY77?_O4j~je&|hH?Sc#Th?M}?xa~~q6SBch?HF5`l}o{'
            'xdGFMS@&@}<pZ;&0z~u*^=@K7-(E-QwxmBtv3%?9uV84wE9DbsU)mpR!Y#s@oZg;J2&j1wo=*OI-~RM=MXV5n(P&B=W)R;!J_#P3'
            'oXW2?{c_qYPRVbR#Pg4^y{Q90K*8J=h4xigl>9jUI9qM#?_G8K{mEqO5YzMQF5Sc*5sT<9j<T_Os-BFGjZweKZ)08@q%&T~5M8(*'
            ')nbh#n-n%Vtuv4ZUOoN$+Yj$+^*%;7WKq_h*%TO%LY^c5LHZ#B$r`O3qmc#BI4-NQB<F69j?^5zwZzHtXjLKa0STKp&e4ao7H(vK'
            '7PurtPr2&<U<Thm!Wjd~N^-`WuhH{|UJtch19Jv0w;av$?5==Lfpp@POP^u7y9;+j)QmrDOLiKz_~byApjL$WbyS+^lk--FLMoO8'
            'a0v?nlof>=K~9bkud5I^a2c`hC0w8)YSVYPzy<bcgpS9M-;KiCEMCso<Og~s)A}a7iDsXoJYyQoZtD$!6eO-b%PyGv?09HNoOwik'
            ')2TV=L`j#w3zA3i$Rks%n!Tx#<Y#$3FcN=TgHsDbsP+kU-jPv=-CHFnhAZGPAO)s7?SLX)I9BIX4(k>P-#M}=b`ynYQeO(6BsQbi'
            'as?coO`OXt;-Mrc-T{~V1e6Cy<mB-W^&$}S94q6%MlzjMz~PGhRtG?r{qY#>j(kWA8Zw7{c<9sR;>qcyC0g(rG{u!NE5pQfYO<l*'
            'VPOqphjXTP+qCV^K=aZnS3Hju2z>=e=T%ZGql<grqV1aI1ug9+Sef=vsLduD+oYua(Be<|l3@}&7FjMv-%a3HyfjK8%LTvn(O#cC'
            '{vN|qB5fQaPE(frkAHxH@^+_Pe@uT1C)h4Fv*P#%a@k`qp15T1<b5k(CjXl)0Xnaci7(=tFpnQx0TT1IL`p$VDB$z3SazqlB+`jV'
            '*hek50D%~>zJ&8+Q9oX6tA3Ykk8w}HL}uk~TQ>kH8%@4E%UPVRvQyyqSpx4~gh0hzz+Bni$o%ikGzwMEfX}iOMR1MoIM@orZ0fTB'
            'sSNn$3@y^Ij?p#w{&@}VD0nFf7i;|jnds*uBlR2&)e$fo{U^e}f5Y4B8BivSael-TI7+h_^cpSJGk&U$Q2i~APSu42uLegSz(4qX'
            'h)&OPcXWSw?^?@f*Ym_?GBM3EC8+E*t{rbP4lkQJZwB?+>cyP$KSO81ruGlUj>gpt4K_oCYll8Agg36qY2ci8^_svgME_Y8UPlsY'
            'GKWK&1~PpD>SMS>TMs~Hjle*`7kH9v&@5R@KlkBif>sHf?v@jMQ}j>OlgZ?Ms1S`aBkdL*I1K!VsY!-g6K0-O1rfuP>CbVsSdD|<'
            '>WJ28MBaC4G(<yhsf*`&|F?M%YzAV(9*0|um>bA%aj2lw2wTwDtD@^MJ_8;Bn&}|W<FLm-=2`DMXBsn<?HW_DL0C>4Tp9O!-G*#K'
            '7`lR;(N009*2Se~yoJ%=%P~7x#b=?U{TL&2TKyQZ@E`$Kk@LcgNF(fXqg~%@R-iEUyz;?c4}{&a&6cvY+uFgau2%RSn$Ty`2cf;?'
            'Q;Su!bz;4y5TEe``uyGlbl;$!JqYPU1}<9{E;{glP1@1WM{0wweZup;vbb&wXzOjutots1ZfWm+&<29YR91>`R=>ZuB30^x&Nu2i'
            'M-W3$9=<ip?9%}{(7t*EH?-bTuotc|{A0E={A>pAnKrXF;%Q3EgpZDH9S}pJuXZ}S@9mwnh$I}AJfj%pVqn`Abx?FIcEDza*6gZF'
            ';-b80i7dRd0}KlF^X?oWKPOl!ybdv<Pu1Owt!|E_gCm^J5oqCCRQitiEY@;FNIjz^abOu8h1dfg1V=m;W9R&)8FFX(@TQ@?4pYD?'
            '9IfbhXVXb*qp=AR*Ug{G2%pP~A%-sE1oQ(I2z0yIB0(Pf+2HZi{~i@zfe9~t@b_b&h=GP?Xx#zdV8r|5>xytqk?G6m1|U=*zG`TH'
            '$nVu)E*_0jB|{!6J-$F|6|9a-E$j_yWq?EwIk^y%vkfB~zPE8dX!&FuE|)Vsnt?sZK^w$z3mw~^5z$P&Hh|Ar{dy4S^;)T!#J7@1'
            '9o2$hPYjl9G2XSh0RtB(O%=5zHlP7YGXM>eeIp1^_Py&!x)_-L^r2uLAj=m~gCnEp)J`qrTnazX5~#?KQFv20T1uI1XFw3>cb=-_'
            '$q<bevn2pP@yK!<;M?(&JsyZP00Qc|Mi&=Ddvbh2tYsN5O2b87QBHxSWq|lqkZqT&O5_O$BwmO(1U3A8M6xId{^@p{F2g)Wd)HCE'
            'X*{ACvVuT<HP#C7Ly&66<MHLaIDg4FH{U7t%)_41xe`mXyrh{b?=nTuDBA#)AwaN?9mA7^hkzQNXGO>hi6%MR8}e|B`4oeOKwsoj'
            '(wIdy-V|1gRTNSZ1Uwz=rU0+T^0fXlxR|gYNJz4qqr?w_c}AlA01y>&_&c;40L*~9FY+$HJP*VI?%gm>JSjC^enhe(z}JXVf_ia!'
            'Iy%0zOexr&Y+6%mR}~k~`qJ@NgLXU!=f$9IFj64DZ!Y?JE&G=N#y0!Qz=MG=aWwXDJ<->GWrVXDxIV#6{~RjV&q*_~g&TOn!yl?2'
            'f_rBYh+Ag}7%a-$tk)hP<&283XgQ5>R;NuY*qnlAfJ3Bz>j7`@Z=zxNW%r`}lo(Ws4(nSyP&Bgm2dot?Ye5@JK7f5fYAxY_hd|zx'
            'b$q=xYs;UGF>Dj*%Y6ua1?H%uW+Mcl{-lnh(eX(`x`QF0p@9fi28q1#i8FtB!=Ny3cThtPj7N5heppTnn$2-3!#s|PLDOw2!iWC7'
            '*XgRV+bB<fl0em+8D)*7Ddc;X4>V_nT@Oc^vg|c9(LP2;KWtEdW;qB!OpMaP^#Q|5{V<+9G%yj{;n?P>S*<7I@18u+d<9#19xiUW'
            '0Q}pNj#QQ*9T&CKc}D97Vj|`3Ew!~>2aQpt8i?wf4QI2Zq@R<l7nc$+c1}}ahq5de`}qVG2w8EuNGdo3*l);DU928&!W}z#Hqey7'
            'k>dpChB2(<2~lYj=9o%IiGi}1z^pmml01qq1%!k^2!Q6G2xF{%#STico00WH1FS*8{Lyx3@Tvf>_-$20OBSdpfzO1xl7}Z*5}>Fk'
            'M;elvN9z#HH8~?mTw44+utV)h1$MefvV!t0*IB8({b8yy6G(TRWlPMqQ<4no*x2l#e;t0!te`>D!Vq)}%gh9R>vqUWM5nYor(Mkv'
            '*WB2JXv5L^g#)E$(LJ?rqp!Sa$Y#H=w0St2M*3}I`&crgXR2yrAx?-rKO=R)>MgJ>WtQ1ox`ruw<A>ST;qKlfv~>Zd@a^wC1>INI'
            '1Y{S_>;!-de=Q$53nk~FZdsGSg-sT9D@4wvta#0QDSLy$&py*a*VuYAXp%uqO?q=V<U}Yk+s?@5b#H7C71_ZUQJv9TnD#y}>oY*F'
            'dGn{Ki#;u=C2$hr+Wd3Sf127DzHA-%P7vQRp@7Z2&54qgSEsc!2D=jyXq!dANkT2tBbX-h#$;SLq%~?8{hzcL({YAc0%e%Q*J-pg'
            'blL#c^Fp&q7)^AI^XZ#!9je&%TO5&oH--TH$?&+_PTnBcGZeo;zgK5pHI`!J?6I#v7H`ca<XWLd?qCfsr70eaCE5B8ud|2z?O|y0'
            'N6~?#Z`AW>h4a0|_>o&6y^!QT8t-FfHas-qlqdp__`uRAow_X72sD;R#_DCsDL0&nPf|l^L=q7j@Ql{7g%(Hn3~o>iV8eMf^bXsg'
            '*MdSIGwaFFIbZ`fb!KW=T>IM$^k}q(_IdszLH}qO=Ug!%Oh$yD(e1JAK6c0CC}ckOw~gI>dtmLYX#7qaIR~6A<g|6(>~HLMYd!ZH'
            'I-8Ho(e?xDOPl`SJNqYpb>B>L^Om?ya&wjuAM1q^-H1qVX4Op?)S;H`N{v=DNyFT)!*uBhovMwO3vV7fma1<3gSU{YZ+s`&e>nz1'
            'xHhfH9!=<<Qlq|FfqGbV+B;wo#b$(5yWPp2c5yl*#bE}sMXz|f0r0yg<4Nr-aP_oj7>jvWu@h5UMGa!CM?x`g96`x$e*!ZiFbrjM'
            'dvb(Cy9R~Bj&Qf?vS?R?UF${c#Z>Pl7^`R5W*(;?8wg)s;*yp-A>a|HLiHoE(80Ega=qc40DSXZy;1O#S>b#z2mqOT5`CnZE#Yv;'
            '@oPIc`Yyuocu+VeqvOeh2dOX)p-5s1AZ0R^Q0ePDik2~)8=QAkaAYi-{BT!DAJ8#oGg-p8o||w0fvU?qG<`63L7*5r8o>N+hVF@i'
            'MDbXd-q22^X};x&VWdh};7MGgivuM<@K5n{hmT-|2-b6qb9?jly}Cy1V=W-JZVVRSi6ZlMCT}U!z8D0q7D|*flUdiMOU)y+l2;o>'
            'S+d8Y&u32{`qZpN{B+;qKa*pdwHV_TTCALY$PQ?QBDD?zZEY6-R4XUO#7|aE5Y3%dr>?vbEwrfYD$6j=dgwumRjHPH+7vlQixd~P'
            '9iz~4SorY<@Z8cd)lb%(aI4U3OOIB6(i1X(E}bk`F9PF&o!yuAs9I;Dhr*v5FRv3^QBW}Ccm)_kdvxBvcslwY;po%<`Cp^|`@fGa'
            'zwHlc2qs6zjm$uAZWCkp>cWXMxOB6V_K&;qt&a3SA3N#P>_#7Aut#gG2YqDs-&DR`EVGMZ6BajECV?Fbp#Rkd81>)uW3yO+PuuLw'
            'KwEhY?y^xLMH4RUI!(qSWjTY7$5xRPXGec&e1&B_IxxPXHWW4ED-7AA;T`?)?)$cdgfH73x5Pw%bXk#4nfM5<uHU~17;+RjU#iCr'
            'a{Y!R($wvWYGd(=KcP*qUQ;4Be9~gUc%UC%b3Q}+B1NDNYK!Srm{AuEYC0LJf1d<_$2ifMWUK@JdhvPe%|mYtH`~qLG6q9PZx(XZ'
            'arXkx>a*RR$R-N8Iq!NNzQqMPMlOQE%m@YmlSWcRrHrP1d|3Vi*}JpQCv;@t9&sXPdOCQNkJy`N=W-=?b%s7w3>)-iD<>Z6hh4ZD'
            'woTR!@wIZqK!=1uo*I|b#oYz`W|n@33+L?oOSZ4U7x>DZ=e;Fi6|QdTktE3BlVNj1kgVLY!6xJ5CwtH!A9NEAyq*l(*B=(~iLKa?'
            'KQmoMABV<Ek;0a&HYD^699&pd7GI_^Shaatjda5B1@&`@#08@0P@O!8+M2^(^!*;;5S})M(P%}F=~EsUZCze4UiC)}7&SG>$mII~'
            'M_fmGIO-4Vlod#tIeutiDMlTqkaCxvp8;D~qKY(j4CEu(nk(1%Z$7mh0+!(olX{{;+he?b&{^3D(Eh5W58r3%Opp19?tF-D)_=Fq'
            'C5)EUz5#lfgN6@wz|Ln|KKoW2(iT>@T2hZgL5wBA4Hu~Eds6Z`nc5vvcG9J7&$kc^Y7(UBaEha~q=jrJj4?p5t=HnD6{0Ys038HV'
            '#nJ$dyV{QFeC$e7<cXXY0~c!sUQp`@`W3@(mPh%?SWfq;-a$j~V}OW03&~a{x+VA3C~@NsJTsAF9MTq6ZQBB5b~?_R!ND329u-^9'
            '>wVSTdc4iHgLdhO<lCmfsR^v1;S9TsIn$2kO_)uC&b0Zc*{sc+Gi`h5Hg%WYLLn;T#5qw4u!sP)(c`X04gKBXz2cR%zaVk<Ay_q6'
            'YKNr5lDpveMgt3wSIqER)i7;70JJ%_PLN_0?E>iU2m<LpbQ*u7p4vHqb2N)pOD=~CLR*444x<s;_hV3Q=3!YTXfaY;!C<VOqKx7}'
            'WQhgf<>cH#xdqKN#(%)MpA|YAbl*rPg~Eyjm>=rFBPYJaj;TsASM$A%!lcldjGW~h>V#J~lc9GcTn`?l3Z*!Nc^Q`&tE<jS;`3M~'
            'nA1>euFDxh_>hW}da$^lCyWJn9Wc9M+_g08u#o6-S(GfPd2y{4izXFx?<g`(5sa4DZI&6QpX&xkzG!lEd2wV*mPeQOAm+7zB0lpV'
            'C`TOahD9@f7?>l(c0v<b_Cau-%nR_E(7siJexJOp9cx8;QD6O+`rF$#AI@JtReyi`HwAp7+=ZWFRe{(6y5T1J6!e2VJkhT6EOvnU'
            'B=!UH97g$2Jf6rs_6f_5k`AL|qpXJ=c5gv(T__p`meAT4Oywl)vFByBilby{0{MlN^*gG0gp}*1{_SM4b!@3*Z?5Ba*gS2I3V0OO'
            '#FmgC%ILF<!xU<XrRF5*UAM$(AE^Px{Od=QWO@{s;)3-YuC9zt`)(a$loq^7;uOn7I5oZELLqa+z${pauu4j)TrG2#dh%Q>EYkIh'
            'xww|9Qp~i!jsRuZY6WU`Ar-{KCFT=PxQH>wGh_v2wqZ;dm8(EDMFD15<ye=_NA!)=iz-P*k_B#Ft+7@&^KT5FN+P!vacba-Sldp@'
            'P2}O7RSHeX8P?aC`s|yX7g>UZFHA+UM|h3_V(bZFG?=cj+$a!MBVE_<9T@BlO|QYBxdt{Jey)Fbi3ZYvt`VnqG#|tu>vOavYIIX9'
            '(Nx2^w*-Ts|4JY4zg0G&MfAFii}k4S&7qZBDBZn9QnQ9Vro(A48S=+NAl0~>_HTev`a@lZa=PkI9)G3}?qL_ipO8BWCQ&I^5@RsC'
            '*s>E?y-FpY>2YA^xYt`&2e1el9s|zn*0az2@5K>{6-Sq+lau8=6>_iRqWFWp4U^7P$G~j5u)+hmq0j;9xbnmh$)#OBvte8RXEWZt'
            ';w+8MgJB~n-Ydo8mSgd!=zaK&dL3>}DY|4gR=<kygn9KtU6uwfl3@10u1G1zfgHjEhu^$@d;Z>36CoB8>hcsQGwzTVQbCYVR|w&t'
            '+9%^<A6kylrCgEcI+nN-eQGS@)hf!Vuqq9zgdS`51|U6PA~phB8mpuKek44a@{V;uy#j?|tf4dZ(0Oq2nq~o;(a9+sc2ad<PmLNl'
            'MDmb})OZ6wXC$p8#D*@~U@=iwTyvQEDF((?=}n4Pnfx&FOTIW|U$EK<gQ#oI<Wz-fC`TY8pRt5ua8~#gTHLZUa!!D%L3MPsx1Ekv'
            'pg$k6%`z>mrCPe`usAgCZG_9is%Q9B8#*dCOdAy-J9$iFOtoV)@$DZz5atJ$y27ol^>9qJ9(uM`2jX9aq6zVA)W23(;B$>deza?v'
            'E9*oe7{%7^V5KK?%1Ys-DC`UYF3%^3rS3Q<1oQa)hs0<6=)Ucw==Huib;OkdRnxM<cyBB{q^~_-!bI>Spr{k8#6_V+7m#oe90-e~'
            '+X8EmLd|%iL`Ij-pc=ZyOCN+7>1GEk>C7SWqBe7!1lcALpoW5R`leg12MZf&xvCP;c{g~J?F!g5Xb_7+lw*+<(WvZ(93EH3ITi#<'
            'Hsjul53gR$&YwMfBR3b!o_~1vlpICpXU{Mk=#LnN3SiC=Kbwq%#A^}<iPy0rOETqMNF}AUWY2f~R7<q*3O04c9)!pDR~Yo**K_<-'
            '@ROyuCK4`)I3YA~@C%D}#9LY>@cP56_b-2b_44c;4FAM>|N7~FSU08|JFnipK6~^2eD?F%JNX2IdvKV)l1*Kk7Os<Q&YCz~zC?FO'
            'i6ug!d#K1R$QsuyY@4d3xP}Ptt}X?_=^M3B4KfCDOUBh$okt+s&Ja{2HTJdOa%Io@FR$MI`RS|K*_*R>fBF0D*Rz*@`RjX%Zu;J<'
            'yh!Cq4y?N9RatzfMB*8DK`8q5uWw(S$x(JBd5w5PSQQO_e)gLD`p6YvPl*!rGj6YA%BfH*e`BUmv@EC`S`_(2%$tTg;{94@rO2w#'
            'o^UoL<$#hI92CAPFR)+2Kx^GZiIK5uC+fP1cao{1r3=<}lw2@oiz=@Ead@r7uP%4kCLjKG)sXx;eE;98A^j$5&KDfYUsp;=3hEYk'
            'J;;|-QbMwy*)}vBAeAk)la(o-)lW*EG?R><Wg&Wk-n~B8{0fhVE7SZ+g^N5ZP-N9x>({5h4A~$Miy)<U>hlPsM^dL?hb?t<8pyOd'
            '&GuAbVlN+5<y|(y9VQHbrRflD7rae**c9hQf?c83Rl3Y^yW`z$(RGM5E9Kt0t?&rk+RPF{+3V4WbEZcyuMsDO!$j!TuLx^2@o-ok'
            '#5^+kgQah|WesFIsaOtuid8z(Rm2si|9U0_&Tzg9Z=*UxfMe-3UV{fOkTIWjUl&6`MNSNH?L?&2j5}Ww&%yzY>DGlts&zk~_vuy?'
            'W>NOV8>~99D2d$bIx18B)D94fm&Cs6uW%vgsZ&G0jGT{X+PPb3A7e!b8f>eR2P>UxlL`>lXbwUcoGL@9Se>fR4Cs;i<2}A~3#`*%'
            '4!J_=I$B^n!Y7sGDfR|B&pQ5OUlT=iG~=vtahxAx4XbiMojhvLgJpEbi$|&r1$s;#ypaM;Vg`S>E4j~sJ%C1_8!|v>czVEH6uf&R'
            '>Jk$n=M`$U$mMm8GrWL_9sdxFLHB_N81O^W`|bpt-QS)3+Yf)VZsIXlx(Fftq>h^dkz*3?9pV9usbu=&$EF=dVKhpbHb*UgU7Rw+'
            'OTbaF203^zIv&=rI6x_cV{&@jz?L(DrmEGBH`<%p+HSHPcwHaM>LV$T<In9<Z(ZHmL{Sz0nb*-&^t*y%USlW-0^f6^JN$O)Ov1^C'
            'ua9A!8gC`DOY(x9%Wj?9oI7uQu1DV4!qB@34y$*7^_d+i=Y<(^J%rIQMpjsD0g2jj>qnE-8T^1YIWq@xW}eb%I8+a%w$~K^Ujq<8'
            'ES;KiVv?|k<{fQ$mZO2<q=&jL5(|W4`Y^>X>g*7fS8!15kmJ<|hVD~Dd0^YA3%e{Cg0X(0HI=^c1FB{1jlHoEyy{1RIx(znyr*;Z'
            '19=t^+Bi&igCOYc(y8~NX&U2SDe^Ti(hk{XW=3HWji>Mzu>F?G!)gGC64lZWt3>Y8gAW5tW(l2+^L<xZ$j>nd9(>U&tI=XsxWl5`'
            'wQXT&bZ1JyZ{#tEDE0!hQT9KhTBAW0Taw!_iUtHFk?gn#c(O4Xi^;8LmRXAvJ57tgmDw_hRc5KWJV{03xnSRu$-f6!@6Py&3Z<`h'
            '<BMuo`3E86%9!A0oD+97l!QKG+uLBU&kfy-h;DY)-(0cl3vNKvb;G{)I>9pB=o>LgEEb8^v(b%$`>ztT3Br>KtT%!4v3Dn+oE%M('
            '4H`SZDhlU9dT4g@EV+*KBwD4<f0B2hk6S3pa2;DOqoPKe<}-_-{iG(nwp6*L6+K;UxpWVT7<Dk8u&R0t)<r8xMuB*UXnRG5_EPz@'
            '0y2q*U@A*@Y)3S1!?fkX!^hlV&*c{_<ti!e6xA(Gy`fe?`*nyobpHGib-VW}K!4mpugSjl)vBb9`lm9!Z`JYekb0JFmkc)BwaNh2'
            '!qo9<&3WwwOkx0wF71~M_kC`J8M;*(<JIFc&DQ#zWi554DPvD+Wl+|WY<Yh>Qaq@YywOO)@~`e>T|nO_=}q6sbh0mng!Op{b?$Xj'
            '&|FX{g`^%tJ<8;h@src0IEx#-O#mBOW!V~nx3p|6EbqoiTP3P&dK1`I#tOx-*(AVy+&5<a)3h70O^II+Je7rZ7>$h3PiLd`b%)!T'
            'VQWp`pk?TQVoke!EofKzOyT}gQY0v#){e}1IAr;uU}ZC+?s#QWhiHn?d4t}=akQ`}&?g)%?Rapq63J8d+qR(FMl>-ur7(X&Lwg{x'
            '9_wjuZz*5~@o*@Xj=7#rT~V@y5Umy3TNkYn4X;Zm8kk;&*aIxSGlQ(zT}xuv9zD;Cnm6U`9)M_(coSMpZB&fTpGgbXcT<170<&KB'
            'i7u*F$CSwWiD}wsp-sol!2Rh*zd4#to!sB0NxTcw8{fs$^e?@9bN=%AS<}BHk!Ouc2Uo~?MsR-BsYvXoG{KK`YB~9KQIl}aGaZJD'
            'Ap$1AD^&k}!UeWM^`BK78J{4BUFetEd4r0|QPK1t5~s!Wt-HV~l-HEYz76%Iz8l4*HKeSV7eBBHZ_yx4u?3`k2>3PaR4%)TH<n<S'
            '0i}+l=<}0GfO)Jhxug|lMT+iS0v!q~OcC<UvcMz+I%l-A6Un%!v`dp0@8x-IT(Gn@b{v!LHX6xV*oFzjTu)yQtB&y_E*H9><XsAL'
            'q5DM(9eS)IA~FfLOX<!g@*x|SQ;AAp9g8EDI!{5Y6<L{|RA5tKBEmeA89Pq3Cxbq{5}!3m{jiVL;<E<S<6z<xb3h_5YEN69+#{(D'
            '^zt@XuglifelF@jh-M&u<6WKS!~32iU0_sHd67oNe-WpUK?A>+hqb7qOO%nu4q6gQjs%R-sz3_hA7u98PMqRUSSoUAeb@4tho+&C'
            'w^H>f-g-P~Xhs{H_F82$ZFQ0%gOceLs<cB&TGqW80@u>OPluD#AiW0smJ{8&<67)q5w$W0QxsEXKv{mC;rgNRyBX2-OLs^IOspZn'
            '4M0DDA?|nZ2gbADqogJF{DD}3N9v(z=stu}zF=*n4&gG&$;~0sqEB%hGK`BcPjD1{OKu%LP+FCde7VrsOw6fePoE()^5eV@JEhXS'
            'fL!5J&-<SePtBFh4+^OIOA=AhHbTNl&?%teV7W|6TL`X())s$jNKNN-;UBgDYi2I@apSdI7}z4n)4^pAhq&bIrsuH!M-SD0{aTx|'
            'T&6Q?x%w!<+vi5N-;H2i^Q?a$ptg&W%*#DjBOOw7xAQ)PgG){Q&W3i(OcZ+ykLADkG@5o;c)Y9;6U!}$Bh9ELCuZUW$g@^9NMVIY'
            'oV|Jf^4*zzrRR|9zF>iLN@LLCHPR<jou<qyy1A38E@cw<7hhLdRp>Oe!O8cN0%H^;tAh-_+M;g7Q;<9u(@oM+LJvh*n9~+7d(>mr'
            'Qngw|cjo#%Eq_Yjo-OxoGsG)^4YBC2A3A7BqLNf0vh|z+l0q=2I*#zwM`C{>>x3)d3QLqBPxZ}FAkWiX$9Tz>2_8nFTZ@epyo@P('
            'ShN>rNW!xwD_zSt(_$}1%oK9LBRWA#?eB>7y^j<HCIa_BWa!W&7fNj_S{4@5sQeVgk_^T5jyzEF^ulrD+`-v~(d1YpSQH|>mvGE+'
            'Z70ZYxWU<fygYyZ^37im`bgG%8*e$s2Sj&DM3SIAwE7SzjWLerz``B%RaBDxEB9n@YOh^zOjj8L*g{1m-3O{ij>H4{Q>{pboQueD'
            'jCcnlD-E9C)A$B(;6tNJ5Sxy$xN{n>P<I0bPHs|jXcDjJWFJ)OXnl5#=#N{eB#9C6qApOTDPL4@W$}Y6uY#9R5ACG5e_*&e?l_88'
            'q?9f6O0{fK#ySP*!uGl{+0%eaWUYPc%;{fAG?7e{YL$^Hj$t(mj6Cp^nBO39%{skw=%Dqe@y(&pg8!nfwaA1*nTv;Kmf^$ku(4?3'
            'xYp*?67P9s(Z!+}oC0c2W|Xb5!t7Z&8MgQRNbLzg4s;FeZ(#F%?SI1F>=da2qvHocgbTiU{e5NdNwJ2A)YS4xK6KAp9UiO$Qf(T0'
            '?2!EUF1+pQ07|k0Osy*ytUwVePSA`W!N>k#?Zk{6qk-Bt8v?c6>tlh!HQ_AT*t(>1Z_VDY=cYL5O}}w<YhWjG$dtQ7G(BkCST4pH'
            '*a4zLYflW-+jIIOXk6)vxfbSfUAD)d>gt3TWBdTHpeVnk^VA%vKjLZrkE*cmM@FuMVH}aVu43V)Xlnzh1TT)4?SVcp6I+161i(8%'
            '#=y)BJF|%st_;5~kV_X?oweaD7iKPE^cYMq7|&)YuKMg=-$3i<dSG$O!KzQ0q2WAClUUe=+J##QrNt^8$#sOH3{XBW_V79ebc74o'
            '3xm&g_xC#8aut(J&%eN!;?6nL6XRcz7XwcZ%f<TC2}^zbXExZu=+OFTT@USMvn&lZB_W1#&n#msE#A2M<|U7i#uXqSy%}uc0#j*S'
            '=55iK5R!M*3-FO%&vig|#@+bj-Wqo^^_~rTdsKjF%d?^5@R&OMx;-hB?&;w%20HDBR=-Wfu@ZsocK3M6*G@_JL$rvKr3rTU=Q;I>'
            '6V*ZIv&Ttg9DVP@gG2hWWS`Z;s<jo~hOiAdz>=;Z^*_*8?zvpF@!GzT&;C@#&TCXOhvVhr>YOUu7$CT4vjrS-GpTzZ%6K&A>(Y8x'
            '{Xf&v>+CR>He&26qNB5=FoXaYe6FK;F06}pFELEyhe>ch1}=0H--LF>2CBHC57(MLFtJ#Ee%27l*VLL->>kly0L7fytm~9?jQ}o|'
            'Oe5&#Li#MF31ID`htE?p*P%7y2Q+k>b@+9^<e?)HT<$rgbzRi@E@r#d?o4hOZd_O0teez&fAzF!*D==~yOx@kvQGAO$ZeJz=NfBa'
            '1AFI*A)?lOrp@4q4witR8{fvORtC{|0d6-(@zBk|Vdiep;7~{&^Vtj4xjNMC;Sg?OOksF5y7w(w^sLQfQ`c(N0>-;t1mjJR8_wL@'
            '35yH79PW~oDDIrpz6;NiOoS+OyZ;Wupc$dA!}n?^7b34;4xNYOHt+4N9M%${=x@y6pq;L~P;XQ0RAT?M2dmAYLc&X<N`Es!px*p@'
            '>hrf0fNoBpl`oODIo+2?+N@Yt+r%4gO5Y4_hQ*zeMf`Jn(2ow_(8J{49k&g*;T$a!Yj5SpakMk#M%#PO4<X{qn99p72Tl<X=YSel'
            'TNFw$+KX1qt^mwi1=Y)1c(vbasQW0<sY|*J^FrD>*OCwULTu0;B$DvHA#Q(-O2@+H+Z2<_jV>n=65n)^VSq)R-Kp@7uUxY;uS>Ft'
            'Y0J_<SevxsYBFp^!`|EQEQIXW=I;GSQ(LEF`P&y(i~tsGBFd7aIb9q{mq(N$X||KE#+lOLthG^1pH@+%^_1MTA8!KeLFMZZCCpLA'
            'JZ(wjXzh=qi3j_NAAFHA<S9h0vM(=*EI_RkNrYpqc<f7g^-4xJ&3wRg)jD3x<J`Vo0adbC^7l}w_hfGrx!cA{X0Bx%{3iMv*}Gc|'
            'ZDK)3Guu-bW)h0`1*@MWNVlqp^Dy}$dz)Jlwmn?TAbeP>;cZq(b4{{6Xz_OyTrA43xKtOzP&Y|D*G1;OU;+=8erqZPcJZa&VXiu~'
            '(GEQQ*{Z^1CK()7F^(r-*NmH*I$u4>krF%B6o@fSmbQE`(=Tfxcj`&md{i$E>@}<vgT5Y?^+%tZ%EP_2Ve5-2rv5EGQ@Wf-*Lj{k'
            'KYQ`?!>jia$Z>92UmUzo8`T}ORAD&d)b*dbr9r2Bw;Dck?Kxbi+A{$;v6{ABiO6>9g`FB|CUK%k*<S1>tmCp;(APde`(K*aPOFy0'
            '2%V?MdEI-qxNM#;0Xo5|duPd46qsd$M&C7S+}Ep!_U-0-?Xzr|8V3gNL(V<_88{S-fv|W%5J`=lXz-f2p02v7c6NE6+p!nqiEt&<'
            'ev8rYFr4<;Jv+l;LJR*s3-WB%k-H$(&yIb2k0qtpHel}~b~|Wk2(3DFT{W{?qa+1#Rbw;xsp+qqG}c8(y<z_Juh4GI_HwY|SV)Zs'
            'F25V!6*KWa3Q-wZq4^=Y6)e({%Oj4RYs6hKg(_;p8^(q&SOzVf)Z-Np_Hvu9XLAat<r7f`ZVo~RDu=Dk+P6A;WJVN}>s_m-8=@SF'
            'gBtj^fp}`=o=9j!o0PKQsgWaG#Q1&-T=uTBG-XN%NnnXi#8NL%tmSAmu3ABX#a67XFB`eB#jRzE+>l|46sO8YKB&+8!v*hubmEDr'
            '5EBZ!FetkmDrh@>pVQaDgA_v15?aa8##j?B+!Jry3b-r$xUC!7vOp^18j?*{CbFW7Ky;z(M|@oq0MsQ%sbH+NP87>Q+L}b;Tfx`f'
            'GzM7N$kX?OAGtAsseMs8tOlfq2rCguO=@X7ZqS`o9+gIG0y&&vyHWzj<s(ty0HkbT-#;>!rsN)0G;E;r-a9SY?2vR~w7uwVYix*K'
            'Q~mtmmX4jU1B(5Ptp&A>)e2xV-2w@f6|SVz(*71s(jV2YPv5<H`Q|UD_C0wRC=zY9<xUN-Vz|DGuVjwbx5FPb_+GEKNYLBl1c#W='
            'wKv%)+p26%-r(E>=a`!UG4M;_G8Yt81FB=0Rf~1Bq=E(JE)3Qru<khp>DHC?`c`FSH6D*m3)W7#g_dZRrb^!<t)%u307X^;h9uuo'
            'Tw=%9Ey@bE^}d$eoMIiGJo#Zpu6i;C;8y^f@wbexS^i&##Ts;@F@HV?>WVs4OoxT(&I&qwIYpJ;;L#c5-4`6=d}>#585~bezNZNE'
            'lkefIpZEKoVS)#vO*#b6N$8UI>*yo@HP96mY_&z=IAz|pY&pYAPdVktMFP;lIc0zl78Iy+QqPGB)SIf;(IUqCAuvFI8Lou!0X0@h'
            'Ij++-uv`{~fMW>Ocv)?>MXL@)<3aN{#{bO1ViCtv@>~unc$}`>VV8&%#*9Z?ndQZF&>y0Xeu^pS5<NM^;9nMw$2^KbzbaRwKlU9o'
            'dP^<i4O;NZN};HqCYNl`YnzDVayg;ux+w}FBySF79LPrD^?b(x%kuOejYfr~tcjjX=ah-s?GH_;RGb_2{FI7zh}XyKGc38W7f#z!'
            'Mhs|H0!Wz<MjD2?@=-$3c2{tQl<_KDloFqZQ5O0Uo?&E!r-Z9agclUzEW=PyYvlTZ=;$Lfc93e0A*vOI^*gsoNY}P!2_}nC{mmrA'
            'V-lHzm9)i?FdSRE1&)6kaphV#jZ?~UL1P%qQd>KgLi%1w3gDfakPD}xIp-Ec1QLdAir|kLyRVL*vw`d_I^KwBEFy}Z)Ag!AUvhxm'
            '8)B!$%B4Q6MRPb+;62XukGg=AG};%;$8L|=yR%<jp8b0EeD>$R1L_NvVSLV~AL|0Vi+B@b<S?N41~x5d#IZIvIo4ORS&-l*aIb9x'
            'xq(&hciW~3zG-=H4*&e@)!DQ6Fm64Mr*HnIw%Ml`=v1Md60Mj8+9!2o431$1uszEq1$fj0rtknanISmUcyOISK0RF6%iF)aH{{{q'
            'L<-h>J@1*MC+4?JdHUJu<KD8y{@BnHHqk9!f`}g`VU85X$vxhP0xP5|aClms55%z)__4_xA;Wzu_d`m(xD3RzAa#~z7SIc7Zv!$f'
            'srFkZ@-8)|eV~V4000915^p8m-~njx;zPTy3p=?>YK?Rsu~NWNWRBo)m&V8$L#KP*d3`TU%G8!ObG~qfaM^3Afp5};<i=F1(6oCG'
            '0H2+9e2{qqs`HFkv;CF~Dpps>-kqBUNHiORHbjD2OO<{HZc9@X!3X%y%#=g(I<_DjP!MqCBj1#P&x#!~U|@Pw*Rvjk?X7Ac$`wQ#'
            ')w-jEiu$<6_`y5e8!E+Cqs2dskYw}v$TXYW8~Pvedw<*aUxc63F^PHcr#Out8}$#p2OD^zU?br5j;Lbbl_R;m{W}WM$yr5rKZ2@s'
            'Y8M~qYhi&xW6;Rs%Ne$1hjlKp3-V#UY`)bR*{RaREGH7WlIV`(wDy3h+sHn>Fi_A2*m5MH<;>nwLodX%z8oMy1z@M%zI%T5?%Yb`'
            'XA=@~gkzMDLQ3S`KnF`Q3tI5*))A#m1FxeLn{-DR<r09a-cbTLB~zC=DO@*c-Zk|NNMWEF@~2#rzfV~dy8`WhMRk2m253yUFZy+g'
            'xXA>{Ru-&y!`C?swXwn^WKdZ~^)pD^k(!|0pfI-zphLvy6c(_-l|ca^dl_A+$Lea8C3xjD;MqaYRWP<qk9I?Qm5zrkU$Nm8SWdXS'
            'jp2Agv!bhR^gDX>*By=a$KABjv}x}r9@~=Rl)iDebK`@c9YS4arnS9(?YC}Tn&$&(e$fS^YY6`P#tbMOxG`{g>VIo?y8E#_-g|wu'
            '+roO)a5UKm;%W2Vg~l;QK7wYxV&6R-0F?Wn>uYm>*)7b&E>}d0U`5t>e(ro~bOic<b%C}I!m|DVe<4>weun1MF-SlZ)%6MHJ+8_H'
            '9_twv+QI|60nHd*Yb}Kzd1{KTxoH9DHhTcLFR+OO9Wdn#2?1#|E)Oa;6^i7e$c7^>G`0#1I+}?y8mIR8-WdgQ>$=y^?ew4*puxAi'
            '+Bb9!bH3^NFs#Z9)+Sz9=j<uRkT*z57hW|=Z@Y2C{Isni`z$8=4H$J*93Td(D30X+@gB$1bBkgH9b9Gvcpga<4+WklT)q{w{%{r5'
            'Mk}q&UtHSyFdR-PUr0MJ0EcWrYm`B~Xh<t6jfxcWvx@9fQeo2Ib)57s?FlcPYZra#Xy}VXLnHPKJ0y}lr_j*d;X`*nAGWWC9&Osb'
            'Xgt~ZhBeY|cz#Y{5$FE3nnv8PHd+Z&C%K7K6mCIuIQq=jp&hB{rigE9MXO;`;pO4St~Q32D>+geya!m7+^S&hj;T2$w>f5Jl~l0Y'
            't`pkq)LmNYvnlX)4#H`z8be}Q>*`uFEb7yU&ks!4+Ed(KThW$&Z@9w9&b?l3>I*#f-r2WpxB`_+R~xm6(CDqEBZowVz3Q>q?_4$4'
            'uv6pbcT2N-1M{2F*FGkiy|uCB!#Zqfd0?My9NzRvjJ})9noYTd>GgLvipq#2g4p>e*t5$qqnz2F*pz4F-I#unWYDpH_HanrQ|*0s'
            'BDrDXMcjfy?ORi|zAY-Nl`H-GTO|*)+14zk?Q{wbG2(W2<5ZGr-5qX!#z@{}hI`1edPpB)9JPyw$Pu;6Zu-F?=a6L;Z1T)D@jVdB'
            '=$?_XL5RR=V_}qH+APXZ5k}@lzR7S_RGUpmxnlbl)90x^02w;1IBo}LmWN_@Ok;0FIYx&>MG_VE0o!59bjQ?U()@bUB(!|c5ZPjC'
            '>{zN6`(od&c-Dz$Pnjf)cUv33YAnof;J}Hn3~ZW*%%MQ8?jK^lJaop!I_WxWWy{z;+15ba&da9TVs)idR2|BBSdk}F_}RHxLZWna'
            'J<o@Oa3Ai)jmtGXH9~U_9(E%>mq+V5f}3vOKO?@y3%s|y3SW9(QBf1V@Q*+o88L|!f+M~tGUo5<QH{jw%Hn0TmD>WOL}i(cw6OV@'
            '1A|MBEew}Cqr`QdYHpy0(NWq=Si`2(YS?0o8#c^q-p1mhLfQvI8<qNPy|>3wGpXHB4KP?h?kQ<WUupLf>&YXJoPM@?=6j1V-x(bP'
            'wdRCw&LT`$lsi?Qsq;VN_wE%+I6@;y_{h0bKPms6UC;MHFU#TM7Vx@UjP}{u(r?>{{Zf0@fSXnNe#`OdI}O{1Yu_rC73jF0WxQi*'
            '?!kAWVs--F!Qx%QM+X(O8B?(c{?z}b=OlF!?{Hx@Ia52_mhT!V6qc7qg3(V^KgkwhGNKw)9LPPoJ!$viczj~7Gl!t6byVdjCNHeY'
            'j}{lfT30`E+L)!f3j9L7di5Hm7>;q${xJ>|*;aEnKGyaS+K!O^Y3s>iwQ>>#>L1S3tW<&ehYf?z76ib5wOJV({so&1v~ks=AM%hp'
            'aiCr1YM&y}`0G9D%CIo>B7<lnI2cItapz##$hm*oO67jqg>$4r8}~um*IopF<C*VPFyX`%1~GO76}S3#9=m+%;NaaKTMAb+Iq_qU'
            'Zxi^*XdYq!wFA9nC0aTd@MYbWYZFL!v*Ege7am?cJb&B3;$>SGQd%4@!=u>-Ke}={inbh-u7`kXn8al3fC*q_cy}ZgQ7WS7X6*J8'
            'w7X1f?f0~q$Y#`@rbN{e8r=AZVvxxz(0CPeg#UWUUxwGZwd}EK)^`)S2cBycqy>gN<KvSj_?Uu?o#)3=-m@QVGFlAprOWrw>I5GU'
            '!-3b>RRS9gSn+HblkrnpP5|t^RkhVExc7ziT3ZESzB#2^=<O1d@tWxGIkB$0T5GWhlVndB)!(f?>z9^hZE7!gmGxf4G?W*-MV1iq'
            'K+V){kl`}imazU-Whs3>igx3W#jvSc{1wi#5nVq##Nf;8BpOL5@2Eb}xkjJ~r7-0)2Pv2G<ocJijBtLZi#2G*nDP^x>RL9sP$-b^'
            'b(PD7^+Wq|cNL2Q#kiN7+KZtszM=27rs^9*CD}ce$!BO%WDDCE!Ii*<Qlh`ER{4M?rNd`gN#a1+l*O0HWVXwyvZ5+BY?{R+V#+!l'
            '$J<>xr-1*$y%Yh}o_8PK%zk<K{Os-Q>6@po{(k=Q{6{r8_Vy+!)*{f_!1#XmyC@%Vu~S?!GbTXJ9Ajw*jby5!TUaHK46_2up;>EJ'
            'ueH5?SW<P1;V~2~tkW3^j48n^9E?CuuCqKY*WPX0->551xmR>Tfhf4efnuJOq)BjSzobtGk0sMyuDLUP2{<bCrOUtJQqH4wcq_5V'
            'c<Ejtg5!^x&<ZS_%hiFgj4$4UjOAssUR^`)!%M2>3o002PQeKnI9}j>Dc_2qV{N$RZ=dV_VJ!$uP#^ym^+kWr0;85-*(i%9!VYea'
            'y5!Zqij;N(s&3Vu1yAV#;yV2smm;{qgetj>`wLRuzoHJ7WAZty5LT<C<W|jVEeGXJoI1lPpCi59fTpF^;U@~WZuweHD5@23K)O0d'
            '_gR?xXG(1yUSi8MX)m>PKxr+{^1i3UmhHv<Mm<d>L(59riuvC2Rak}z=o(nHC0>kZTRw&@Hj+dxsb@?3bRBB`A?b~LtENuM1U5ok'
            'afDJUtHyFrn9Qxo@exMB#Mh|83M^gVnV2Iq0dW_24QDIxsHMy_)wNMC>7p6F9mZ657EnfYAGD>pP36p{`+Vw}mx{l$taX}6Qjj09'
            'v?$_KM&$H>=?$j6F-C5Yxx&#DCfRirp|DQXo-x3Fysw}k*(#h$J%?xMZFBTa9ew!w(|0dLVZWg@#jJ&8mM$wbxXIEAq&`4ev5iYb'
            '&Jrw10|&xi>#`-qO_nEIz4*x6)JH|QjL_JF6}<p6|G+)?EpYdBib=Bt#zMyFj%u*SJD3w!G+l%@KT0(YXniBdZ9-=({?8*UVg=~i'
            'A^y}RDpSr<*dd_tQR{B!qnDV29!y-~DA?Ry9nKLus-@XnZ*RG@gW&%bWz?MNIZpj3=Le@J(h?m|P*j@T%bF7MmQ}w4=cb0txlIsO'
            'fH9F$9v_JMFD?l8wT0mJ7l3P=PQmc-x^BHXF4}N7BkuuOMq81ML689SEkKbe9v@E%!}9dCkO)G6QH~mr$1@SRw8#{F#ETIMtJDBz'
            'tM*Y=!+dj04qIn*kQk{fE(>$78CB&X#YS5cKo+Pd&%WkMB#fxH7-M~hcH&T-Of=6o`U)rNQ|Y)T#2z<z{S}2&aN0hl*|CI&Vj9d^'
            'Mlt@1D(FTJ2f2wo3S{j8t?-H5dW6QdPKAv#!5cL2lkLK+bxxrTu3Xb>^|+&ck!7fFo!n1$=#{IpVzgYdqmeZxyz6Pu3U;K9_gx_6'
            'bb{Em%(czQ1y;oR3)}%npnRXkGLKhyr3NcaAy#+WjzhZTR|#|5ff7SkY0&9~#-7lrB`u>@D}c0it~CN;T!AD^d!Uf3D+&=f*v4%E'
            '@~#m##Eql@!dXT21nR5j%uO0Z*B$H4_xwK5``$vlbv3e&O1nk<(rVF3w$l38u8c$%(*98xid$N1PnF28uMXL1CgxpD|8JXOrZ(kR'
            '1)bWg`>=mzSZy8j47>p^cjiBBM~f}3>Wi=2e4Q=z+WH67V&elHr~#OAT!e6F?Lw4mKq|*nGfF55`v0nB$o>^eko^l*8+({Ue~-#z'
            '*6XhNV-G6C(>*?ze}NKYc-cvQ9hD=JH%9uX#i~U6kKQhNA_1zuia$n6xj|@T*aAWgzNoet^Wf%j_E3>CyS2shTbelzO}m-J_*<I&'
            '9wpI!=MprXg|5H49L+&RuK&eK(>U26Ez!PS(UI;fj*#A=wNqD{kV3f(NT+BFc!;s|D*1osXcVTaO~Q0%n=9BXF_0<NR*Qg56hN8F'
            '!WSgiB#)vnsR4Pk0)>GtOv|%H#AcKbaFAe96Nza_cP0sHuQ$<-<BcRP!b#(>y^U^gAvTi;Rl?%lJP90W0jJXr?c$(jP*%jLI{FMy'
            '?vLCsVbtMG7LwC{NAim~hj57~_|q=Cj2m>)8SQsub#O#R7m_F_4AlwVab<EH2cnN&idiDOa$xCD9Ui?!34=qEBj~s_tqSPmomhTs'
            'zsXWhwpo9;k@MmX`LwO<zLN>VpSRZz_Ltd0=EY|i?EPgkA%?2cI%zR(Am#r9Q4JHV'
        ),
    },
    'approval': {
        'sha256': '119794218b882b74fe3deb51c7ea08faa1cc706fe6145e48bc4a40116a202d9f',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;OTXWmSl78o}n1ehBtfH9O-Ad{dT{=~YL?_%RlGl=x^{P}P2o6bvAi%}N3a!$A-><u8fB^_f%4YZY!IprTo}RvZeVKth&$|ec'
            'G>O6>R=YGWPFFh7Sx`i2qSoa$NKS*@E=#}Q^(rXzu-E(Tr_bu{r^&7Q<>vC!)mV-Hb8>rk+dJ+2(>u4bw`E?axmJO?9e+6gcz!ol'
            'd7*czSO<k#V-cO?>iqrP_#+-^RpvUw<6H${Q3i3mSGm?nk6sxL^+E+%5iK!bs6GUF9wjR^%fl?%6;okaFV60=QqNSB6e*KkmPyEa'
            'qh))RQqJ#Ws@m!xkq4H%igNH8Y<F1Fvy?p$m1)6ND~K=d@m9i2GjCC$PEWDnWtxFfvKMKpFfHQ+Hugm;45VR0bw~5CMv@j7sjR?z'
            'B=7+d!n;6a<$MmAFe6aIt!&6qmrjao4<%x=IWcEtQ4bmJ_OpSS6-7=zVXZ}Pmg!`nvuRmGu{>SqIGVFqAoL&%HDq#qQ*i~UtjaCc'
            'qUqRoqS6Ed^%tG>^cKv5;+IH2PBZ=AWt8c~O!;Z1X3HRo>6dTSEK2fnxs1XHRHlod2=KHwTYyBI?uhhs7YCtUr!hF2^`YG?-KL_p'
            'RL!ggW|i6KKf<`oqc6nqT4#FDBi@*=$F&wUk%mZo<!%>8M&q|XouB^eci*dY$-nA9NZDbU6tq2TMRuYW5cdz0>&waYkLvuxhmSYE'
            'o?ofE8+CDWeK)?oJ7W4`|6|ByUkfkOVVW)SdJ|K#+%#VYU<K+SP8QKh!(9wtmW|$gPqykCLoM_l1;iPYizrn~NEg3+i56OZf!Jo@'
            'T8EpdUEJ@_K<X4Sg`{$^R<qC&0&bZ6Nv^^&%SaA}!GX$COt;1aUki-K40@oNl}SPm@9u5~3i_r;Y+3Y>C<|vnS)^M`580)o1Z+i%'
            '$^&<_)3nTcmAqgo1-S_%kz-RyfmvaYWqTqE4Z*t}gNzh(ofd0)NcvahE>43*K2W=I9!L3F=gbss1)C#pN_6p<W*eYN_9Y5|Yo6!z'
            'SPawYvMkC>Pp2x{?$V459iGV!l=ph(8K7wmKe505o~MaSr`;40Ai=z`zeBP3`)itL8Mg}xB4CGoz^}cYv92Gk#?zl+aJTAA`5r_O'
            'YKAxRi^Ll!PXtdteM!MR3@M{uiVV<Iy%PC*{a)`IKnzLO(osZ5P~~!bF}a=GTvwVO2n+!R7g10>9l?<Tddw&Wf?|_75n+SXVU&}D'
            'Mn$gUrSiAwLX*jqS%`nPDY<pd#+{NsB&!&Y)3Vr=1*icP$-;zVQqwY8_1WS;ukuKeIWHJ&;MBBfHKU!}dv<y6VKxLgak^!s0ks0k'
            'jJn|(>uT8vn_IL4)+OLRg#mJais2dp>=-kT({KY62Sot5m={>iVaW!S7h1$kvj}pS5XrB$Y!}goJ*f9$e0vKdnSQuBzn)t0Pk$bN'
            'hO+C8kkVFgCehL|(-2pd&VhoVa*$HZgPivOkWAw!+>eMeLgj=I8S#ghH&{b1fS(p6hrlVtpFOEt!6OR{HYzqqxkE_R6i9}>>+!GS'
            'k5k7u>4(YX_#+wUlea=Z@t%p}bF0E_uowKLh;Nr?EGC5lG$$X&@Y1{h6I)VDB=%^Ak`g6z5TFt8n3qv(unHS2lZvnbMAAUuO6Syx'
            'BcNFMih5*wh=Y=uE!JtSU3yUhv&<!Hz)DWcue=209@kO0R&(rN)5;gfU^C83#1a!q5g3?k76hcrNqy(nmx@vpti?{nu^1i}Edk5-'
            'o;4ir;obw$MZ_TFJ-ov{f0?iy88c@Of&g>%K8SPR*~jsJe1ecKrys}H(9{&c>T+@iQ)5u+Dg5yY9j-a`n;L5+vtYe1ynzg9<x}=?'
            'CCV4xkh@|p1)@+swXoC!9Nm1J{5ZKjznY$Zx}4nnMFs_0rtktSX6-P9wR#J^L~;_uC46ye1WbvQUqjm>3X8rv{RjUr60)8*Hj{GB'
            'ZT6(HN(B^mjIoM=3SopHn*32CejqJ*Iw_N_E>1@3o-81L7^st4i+KDr91frHP;#Sb<J~>pJv<MY;5m}J?XK7>Ll7{P7Z!SNuCG2*'
            'R#-4_iBSfwA382*82kaT(qJ=824-o&icEi@cQltmE#fD;M~7*Pn3OuTsSMDq=6_~ho+=%H9h(ZOWgM(3AK-Kg5JC2DsJ-^b-$oL^'
            '&i)~GJ}@{}yJ)|Iu><+eh_x<XWHHwiBtQ*o;~R?c@qCgC$BW8JvC+JYIgp)l?u9Lr+rM#l$g9>YZ(vT5fjbU3grbrKQtD`gXpNNr'
            'Wuq9eQQQMjAK*Xt52ERsb8@PxURvqG_hfTkpTfe^b6+0T6YLA76D=K5GSawV-&QW9#!<CX=4r_J`NDtEkNd)@Rn}CZ?wM*bTOpy#'
            'Rjp7O-yk5f9L#j**}y-gEagxzl7F-W)0ji)Rf05>Tn3;L20LSS*3zd9*8EB=uR&im5CU&;iR>WFqE(b&i=6XM^KxE9MXYnLnx^3s'
            '>NxcCIN_8KjN=kOql4>_%Srv1WkJ&#YUjwy#3c7I7TGZShCwg|(ffgK9#Q4NI3sbe$BX7vDg;TW{ezJMc8z|`&o}CVb9%`%v#oJu'
            '!Bjz7L<7oA3W})Ax<#76%~Bj09WppIVCB74Dld?by^?n8`K|^JW;T~0A-$}gmsFH-JX25A1dm60BO{smxuYEktA72Y?~FTC=IC$>'
            '&iD=k^#t(zj#utVO}8C2ol3?%n=C;p@F|tmKK(U9RLwXqz`s>m@V<;=<s;cb#T%0U_p3I-TL0lx?g34M?odj9x_f{6cg5BF(1NxR'
            's;MJ~vbfI+6Yh9kf2flXO!{6~EKmRL_4`9CAoTryljqR(@TOFV`sw`l1HPSJ2lZ^`ziY1YFV3%Tt|u4gSL)wyQ3djn6n6X^RVjka'
            'M(=a35%L{U>AY_;h`MQ*r&25-<e`?10PkOI$<Lj!sxl1X$6%kUJ*BI8^`5C2p=El7S{`_o6PH+7QNYVc97t;*JYs<&Z%T3UG0Lf%'
            '$fazisovKMK@={-qkLn^J~HS45|#3glK5oND9|9MD2`A0fm5rwAy!ne05*|;hl_H%%WG7JGzS=xQ`aEK!zem4$q5N@N+RSwB!1Vs'
            '06`<o@-yEXP!t(?sB@vo0C~e1^d%;Fp!*k0#<2E+bx=FVr+&j<F=iqHdubZ68F1F-N%(z?kY=Vp6JDG;5_1Gq04NjzH8+tgT$jm)'
            'zzK@Z{5aUo7s1Gk98#(3fB)Tg|MoqkHHU9B!AsZ}mOIj?&y*SxGw%-tWSehXs(=fBgZ!8MexN$)16Oo5$?xbR+J1%UmT*w{V+VhO'
            '0TrofozSM4R<WkS*XGyEfZ1HD@xw6*<g1pAWJ)i$S_Zb8VrZIDx%y*hx-CU%;h6DtOQ|s)84o9wkKE1tZF2V$J&rPJa)IC6%&sf$'
            '#%!m9&&|vCldG{<mYHX&mo}G6N`6c;z?maOR}fLV;#U-HdYom98K(7H7cWTJBJR)_B?R-hR@Yt^%Gbk{cz!_QM=Eg23Ahmf0B94e'
            'G#Me1z7haba9J<85I2aZWgz)M6G0GyNw+oxw>y2cL1(lPXCMC3n@0#)TfN7Auor;E2C6Ev5)~xT{9E>{{JGx55Nm>GZQ9R$n`B!i'
            'D4A@x6>APE$ZjR!FEy|FZZ6bH$Sl2q@g&MyFv31=`trf{57Lx2=R<;I6@^r~suN4~gxq6Qs(re7haxK2V`LBp8#RZOdWS@#Jk|(T'
            '8{l&mDIhsw!d4^y8a4+XY}76&X)31~{lB^;!$z_Q)Co~MLF5PxKeQVJTKMXQx_@v9eu<E$S5dr(vZD+eH0;h9>^3?q#q@6KO_`<;'
            'qeZ(=bKZFevT|PjX%R_>2Z^2#X=vSL2nt8`f@~nv#)aXw8>qAYu1oK2(*R&)(dJDC(3B_VjSeFnf^_=xZZUFKo$)mA$n<wC)H>Ka'
            '5X7ZcDgm%V%E0f)X-;`nDkfmfr#{Q2=EEI2lgK^O4>h=@0*7F=dVLO+nMx=R9*628%`$1cNhb+TPwjCU&dYHV-z69a1-JO^H2Y)6'
            'aZocA<BWoL&^9@KD&_k9Lp3%m_2E!@crel!rxB0AXd3yaZPYX)Ydtrtbc4!ctG@+hL8W<VqBWAxO`4SXHt4cYN_?Y2>4mY!!2z5F'
            'aj_4+MheAP@;2<+fdDdQ>P_4~c~r_hih#s#>4>R*1{lxep{^eyLVHf3)nu6uWRa9&_&iNx-_9|Qr<9^Q?Tdc@nRCEL?qHG{9=ZsV'
            ')`Z!xgW0f)+3<B{yZ8;?#&1aclA%1j5oC?nJD{OH$E%G_x;&ISAGX7)SBD_$Of!*|4j29Uaeo-)QWU%vsFpxo#6=X=8I^KrzDK};'
            '-Nt1SY=cFFWTPgFG{ie52VTwLz;q`mld~IqOLvkJz-c|L*#mOOxf`z^cW%66(<c4w+;2^--fzcVokdNpA!mhiO?Adpp!Ah;qj4Nz'
            '>y7I4l5fZ)S}*vrhg8Z{buqv*vRzw>bFZCA@gXW6?WO;%!*z-@0(FFwxT<Rg_-$F#SZtM3<+I5$rPIH<)k2$r1JOTD1IA9A(h)^T'
            'j@rr><Z-{*MRKl0V8%5vo<=vw#!9@CO1pl^MG?7CdltW-1gp_Qx__`uAgOltPU90hyl*o@&GpWzvDRR}MZsa0W>?1@Ko`1O<5VmG'
            'Uu9`?V0T25VJbd`9Jm>cd<j(KY_uVG&JXy40NKy+Bi_-=+hO^?z;QE50#Jfg9PAMh)yY$Xp%cr{N&opo?a~-J+61T-Gvu`pO&nBl'
            'ZyS7V1xn99a($@pguwoRfXo$%b}}XvCY_yeXNV4oQ1E;nUoz^O`rU6@1|7lebEM+MLCRygK3aJb^~mR;bnnYECpvsuKd0<qM-Ud6'
            '$M&pJR&(6r8{iS&PGbKhl&jCMsG_$0Y0f1d*K{way1h`HxJS7|uRi&%Gc7tIh4x-NIc-E9dUJ0Y!w;?Q^XqogvYnQj(s*w6HWuk0'
            'o84)7Y$fmM>_y#NpMO^4>mSEgR}*R{vPmXTH(zFh);DQh#xV-8QAa+u^qx)z>SXwP8YRMzIhB3rKM&PM2^~ZRMw@VkMrj@@xgesd'
            'ecPSlZ4wk6$i4R^O26nd*dY6_*iZ#behVHKbp8WUaB1k3+5+9xUIP_4`WOvQt?pXK-vi}@B75JgbvI4k9!a9A`)u>Qp#sZ922DTH'
            '#TM@{qRFj3O_$*eFEa{45)jT#EX5PPM+uYXgYHNgQ?nyH&Xl=u@XBpA+Y)RZ*av{Y-<V0|(|o=MVh_3`8ZP?7j#d-3x0AgQ{j$u1'
            'SU|uWYSj1t+?zQH3A$Wm;@bo&FGK19EX$ZGe8U>S5r^#R+9lZ8!V;*9yq8wCZ1xd=r{((R$i4B%FhLhinO>ZxbSiCf2m@u3YbjqY'
            '59~6&L0aEK?2}s3dBK--ilI6u9mPTqoUf1o=XBC-u4iTXcP`Y;4&Gx38Q0l~V8{tm?!E%Vyao&Ld%hy*TwY*mey(3S;DmPGes8Gf'
            'G(=egN3ISzZA{)C6ik<?$|8$a>q4#q1d?4fFBbi&D@Fgzc40Gbc769aUC$Y&zNT53#GoiZx1qKJp%MX<+64Or1@3T*f_K<l>ln85'
            '-d1IMN9J3GhUz8>$#Fx??YwmvA~Hs(is21#w+PB^rw*$$FBnggs*%6YE4sN#segvBpio^43Js(`giT^Vs5fBYkX*JXS7lNrEQqMt'
            'ku#OgH|oON5BXvI{sxXvg=jv(mo$THNE#^Os|+lBkej!X`fubU);*|al&UVo#u@;SJFsx#oH>**%965@;p+*~|7BkAhK34<YIyqP'
            '_;PY}IT_iL0Wmx3t8jw2iK8l7R^faRk<PjTxEt{8G&n~im?M^V1ZorBC9rc(Hma|YBFeMcyQBHh-eX66w~4Mq_^zq9(%_7~BD~Nf'
            '@j6{>^+u7M@6#;jgVDoW((kJcZ8<qe9pqAtL`tr1-can2@nsSa#l_i*>txTG)Dg!W(jocI?+@ejQD<Q7Z)Xh-?DMw*#PZN&1PoZV'
            'YR(;U_I1<frA1C6<v$%^@P70!-@_P&!+uK*OV5HUjPUl;`Ss5e3l4h(V*knAhC_uKD*2WaG*lDeK{#AQU2u}8Nd^GY2|`(7dwVda'
            'G#2Z4*9CU4zzm*XF^^3V6zpaX)ZhPquO0ui*8WPJ*b<>zHXf@JNuC*edIOQQipHLs-@+sKUP}|}?Av?W&Y=3Ej`5vi=>Bz1SJp9f'
            '>B&iPwMww0F1bn15O1HH;QQRB$8)FkUEyY1rs876t%t$k8_@JdZWhorCZN8n{g`2xq>sM+o?%!PA$8VLpaZgR-)-e^5rB#OFC|&6'
            'rp%GgV5oEJc#qqnF{MqF!<mEp=9ex<i<_>#n*Ac?pk!#23i7#Ax<x5ZP**7PMVY)cteG2{K~ZP?EUxyw;HtJkA<1VR^n=*L$+LBu'
            '>KisLm7DmFU6|`=)?$3;0J=U%>9=XgJrsV;KMSpjThs4=xJ<IpESp}UX1D}3D>!rEM&*Q1)f(@J@Y`<0)}I08>h|vZYRpP-`yPBv'
            'maO1-rBRg_brDwIHKZe}^Q+q%y1ry{ngyj?;$rou8)}vE#}IN!h_4pVRRQ<%gv+g*a>0MO+B!XzhTj*ykdT(Ewev}oeAI>Cx4DkV'
            '3`Ma<c+ILn?ice%7miz-%SLi{*ZJK5%m~w~TfE)PSBkn0MW?$+>)m?|sH$oT#??s9ol%%c-QEIgd3w@E(b4zXT}vU~>$yu+Tr_&6'
            'm1`ZG6STmud0ub~URouI(0*~-tuo}WNrC91jw~m2Ky~KFEv8n8lEgu)xyelVm=6^c3V`S<V&NV^KcHf&ck~$A6=tK4ZGHHSjqBIj'
            '(w$b3mqD_LT3OXv=fO@wFCU(O%=3#@K$ne$T{j`L8tO;}U(=MyEPfncARU?f*T^zRN?5?ZCA}gO&bJ3i`qJ}l?vR{4SuUzv4miP^'
            'KC`J_o#hSHXOpp+>=R|DA@yhU_>KD2!H6=khzd@xZ|+q68Lep{3GR)WxW}y(LyobtYHHW%24J=RioyA4z!^{1_5axg;%nNiYs6Z?'
            'f~ulXEtzr-4F_)N=(JXer_D=MB4z6BF7ffA@z{a;E2t;X-GaP>&2uKJTu(EQ5mAQEa1lq1>&130PLeu{!FH<jV)M6tYumgp1oBrK'
            'z#q-W(1r79srv2w?&7EMrCf^?R^__O&8Iu(R!dY=4N|^di8?2@of2+p7@{VHmM_J{a|rX}6n0%*5VWo$^{@RHQOw>s98nX@$><_7'
            'b!+XR&a5HNnYooo-`2}rI_GP8{ydZ%m-{VhCQ)QYA#2u*WBh?pP@NXZ64GJhj};~5__l6LQX=Di5y^*92cO~}?RUSkHM|4+QS<9_'
            '>F-uwxN@JpO6A1q3=GEf?I+*BKImyST7WON8=YZwyV2IVRnKux_mhtG9A9f?SoCU#z?XZKsyYMwCv-C=*B9ff@h@Zb^ZDn?Nuxq`'
            'dKtVy|A|hAk8sw1p^sqzMRa(yZ$Sr%PVeHNLaywjvqH9S@4be*w2M_c?66pM%)uMEHq<rde^;y?5nX37`dW4SYyz&hIQ9K8-yUkT'
            '8m<*qoB!rP(3&m^O#vfQu5HZ~`eq?pl)G5p3vScj4;8dtqzQ@*WCo!gF(q<#{=%c``qrOGQ1llb<W$(N|1W@6T}_72my@kU`k0sm'
            'V+XNlKH8GA5iX78<0&~}q0=JgKM=_!!>F#39DXO@4XQ6u&g>@;T^|lqn=#*I&`y0B)Hnz|x}o|7Nw(knA1&#y3j'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
render_utils = _local_pipeline_modules['render_utils']
video_analysis = _local_pipeline_modules['video_analysis']
approval = _local_pipeline_modules['approval']



In [ ]:
ffmpeg_bin, ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
print('FFmpeg:', ffmpeg_bin)
print('System ffprobe:', ffprobe_bin or 'bulunamadi (probe_media() ffmpeg fallback kullanacak)')


def find_first_existing(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None


## 1. Girdi dosyalarını yükle: `hot_moments.json`, `selected_script.json`, `selected_voiceover.json`

In [ ]:
def load_json_artifact(filename, override_path=''):
    candidates = [
        override_path,
        str(RESULTS_DIR / filename),
        f'results/{filename}',
        f'../results/{filename}',
        f'/content/results/{filename}',
    ]
    path = find_first_existing(candidates)
    if path is None:
        raise FileNotFoundError(
            f'{filename} bulunamadi. Aranan yollar: ' + ', '.join(c for c in candidates if c)
        )
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f), path

hot_moments_data, hot_moments_path = load_json_artifact('hot_moments.json')
selected_script, selected_script_path = load_json_artifact('selected_script.json')
selected_voiceover, selected_voiceover_path = load_json_artifact('selected_voiceover.json')

# Kanonik sema dogrulamasi (Notebook 01/02 ve master notebook ile ayni fonksiyon).
schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
if schema_errors:
    raise ValueError(f'hot_moments.json semasi gecersiz ({hot_moments_path}):\n- ' + '\n- '.join(schema_errors))

print('hot_moments.json      :', hot_moments_path, '| status:', hot_moments_data.get('status'))
print('selected_script.json  :', selected_script_path, '| status:', selected_script.get('status'))
print('selected_voiceover.json:', selected_voiceover_path, '| status:', selected_voiceover.get('status'))

if selected_script.get('status') != 'approved':
    print(f"UYARI: selected_script.json status='{selected_script.get('status')}' (approved degil)")
if selected_voiceover.get('status') != 'approved':
    print(f"UYARI: selected_voiceover.json status='{selected_voiceover.get('status')}' (approved degil)")
print('Not: onaylanmamis girdilerle render yine calisir (inceleme icin) ama nihai durum')
print('     review_required/failed kalir, ASLA approved/production_ready olmaz.')


## 2. Render yapılandırması ve onay bayrağı

Tüm render mantığı tek bir kanonik çağrıda (`render_utils.run_render_pipeline`) toplanmıştır: aralık kesme, altyazı hizalama (gerçek seslendirme sesinden `faster-whisper` ile; başarısız olursa orantılı fallback), dikey kadraj + altyazı yakma + ses miksleme (tek `ffmpeg` geçişi), teknik doğrulama ve onay kapısı değerlendirmesi.

In [ ]:
RENDER_MODE = 'blurred_background'  # 'blurred_background' veya 'center_crop'
TARGET_WIDTH = 1080
TARGET_HEIGHT = 1920
PREVIEW_MODE = True  # True = hizli/dusuk kalite onizleme; False = uretim kalitesi

DUCKING_ENABLED = True
ORIGINAL_AUDIO_BASE_VOLUME = 0.35
NARRATION_VOLUME = 1.0
DUCK_RATIO = 8

SUBTITLE_FONTSIZE = 20
SUBTITLE_MARGIN_V = 90
SUBTITLE_MAX_CHARS_PER_CUE = 42

SOURCE_VIDEO_PATH_OVERRIDE = ''  # zorla belirli bir ham kaynak video vermek istersen buraya yaz
print('Render modu:', RENDER_MODE, '| onizleme mi:', PREVIEW_MODE)
print("NOT: onay bu hucrede DEGIL, bolum 5'te (videoyu izledikten sonra) verilir.")


## 3. Render çalıştır (kanonik `render_utils.run_render_pipeline`)

In [ ]:
render_manifest = render_utils.run_render_pipeline(
    hot_moments_data, selected_script, selected_voiceover,
    hot_moments_path, selected_script_path, selected_voiceover_path,
    render_dir=RENDER_DIR,
    render_mode=RENDER_MODE, target_width=TARGET_WIDTH, target_height=TARGET_HEIGHT,
    preview_mode=PREVIEW_MODE, ducking_enabled=DUCKING_ENABLED,
    original_audio_base_volume=ORIGINAL_AUDIO_BASE_VOLUME, narration_volume=NARRATION_VOLUME,
    duck_ratio=DUCK_RATIO, subtitle_fontsize=SUBTITLE_FONTSIZE,
    subtitle_margin_v=SUBTITLE_MARGIN_V, subtitle_max_chars_per_cue=SUBTITLE_MAX_CHARS_PER_CUE,
    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
)

print('Kaynak cozumlemesi:', render_manifest['inputs']['source_description'])
print('Altyazi kaynagi:', render_manifest['subtitles']['source'],
      f"({render_manifest['subtitles']['cue_count']} satir)")
print('Render dosyasi:', render_manifest['output']['final_video_path'],
      f"({render_manifest['output']['render_latency_sec']}s)")
print()
print('Teknik dogrulama (passed):', render_manifest['output']['validation']['passed'],
      '| probe backend:', render_manifest['output']['validation']['details']['probe_backend'])
print(json.dumps(render_manifest['output']['validation']['checks'], indent=2))
print()
print('Onay oncesi kosullar:', json.dumps(render_manifest['approval']['conditions'], indent=2))
if render_manifest['approval']['failed_conditions']:
    print('Saglanmayan kosul(lar):', render_manifest['approval']['failed_conditions'])
print('Render durumu:', render_manifest['status'], '-', render_manifest['status_reason'])


## 5. Render'ı izle ve ONAYLA (ayrı adım)

Videoyu **izledikten sonra** aşağıda yazdırılan SHA-256'yı `RENDER_REVIEWED_SHA256` alanına yapıştır.

Render onayı yalnızca **tüm** koşullar sağlandığında verilebilir: teknik doğrulama geçmiş **ve**
script onaylı **ve** seslendirme onaylı **ve** seslendirme-script hash eşleşmesi sağlanmış olmalı.
Herhangi biri eksikse onay reddedilir ve artefakt `review_required` kalır.

Sadece izlediğini kaydetmek istersen `RECORD_RENDER_REVIEW_ONLY = True` yap — bu `status`'ü değiştirmez.

In [ ]:
video_path = render_manifest['output']['final_video_path']
try:
    from IPython.display import Video, display as _display
    _display(Video(video_path, embed=True, width=320))
except Exception as exc:
    print('Video oynatici gosterilemedi:', exc)
    print('Dosyayi elle izle:', video_path)

render_review_info = approval.describe_for_review('render', render_manifest)
print('Icerik SHA-256:', render_review_info['content_sha256'])
if not render_review_info['reviewable']:
    print()
    print('!!! FINAL ONAY ENGELLI !!!')
    print(render_review_info['blocker'])
    for condition in render_review_info.get('blocking_conditions', []):
        print('  -', condition)

RENDER_REVIEWED_SHA256 = ''   # yukaridaki SHA-256'yi buraya yapistir
RENDER_APPROVED_BY = ''
RENDER_APPROVAL_NOTES = ''
RECORD_RENDER_REVIEW_ONLY = False  # True: sadece "izledim" kaydi, status DEGISMEZ

if not RENDER_REVIEWED_SHA256 or not RENDER_APPROVED_BY:
    print('Onay ATLANDI: RENDER_REVIEWED_SHA256 ve RENDER_APPROVED_BY doldurulmali.')
    print('status olarak kaliyor:', render_manifest['status'])
elif RECORD_RENDER_REVIEW_ONLY:
    try:
        render_manifest = approval.record_render_review(
            render_manifest, RENDER_REVIEWED_SHA256, RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Inceleme kaydedildi (render_review). status DEGISMEDI:', render_manifest['status'])
    except ValueError as exc:
        print('INCELEME KAYDI REDDEDILDI:', exc)
else:
    try:
        render_manifest = approval.approve_artifact(
            render_manifest, 'render', RENDER_REVIEWED_SHA256,
            RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Render ONAYLANDI (status:', render_manifest['status'], ')')
        print('NOT: onay yalnizca bir kayittir; bu notebook hicbir sey yayinlamaz.')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('status olarak kaliyor:', render_manifest['status'])
        print("Sadece izledigini kaydetmek istersen RECORD_RENDER_REVIEW_ONLY = True yap.")


## 6. Dışa aktarım: `render_manifest.json`

In [ ]:
render_manifest_path = RENDER_DIR / 'render_manifest.json'
with open(render_manifest_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_manifest_path, '(status:', render_manifest['status'], ')')

render_validation_path = RENDER_DIR / 'render_validation.json'
with open(render_validation_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest['output']['validation'], f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_validation_path)

print('HATIRLATMA: bu notebook hicbir yere yukleme yapmadi (not_uploaded='
      f"{render_manifest['not_uploaded']}).")
print('Yayin, ayri ve elle onayli bir adimdir.')
